# GOR v2 (anti-collapse) Hierarchical MIL — step-by-step notebook

목표:
- 진행성 질병 bag-level 분류 (Control / Mild / Severe)
- severe-specific 해석을 위한 GOR 기반 hierarchical MIL
- **beta gate collapse 완화** (spread/range regularizer 추가)
- **best vs last prediction export 분리**
- **donor-level 평가 자동화**


In [1]:

# Step 0. Runtime setup / imports
import os, json, math, random, pathlib, copy, time
from dataclasses import dataclass, asdict
from typing import Dict, List, Any, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

# Optional metrics
try:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, balanced_accuracy_score,
        accuracy_score, f1_score, confusion_matrix
    )
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False
    print("[WARN] sklearn not available. Some metrics will be skipped.")

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = DEVICE
print("DEVICE:", DEVICE)
print("Torch:", torch.__version__)

# Integrity test
assert torch.__version__ is not None
assert isinstance(SKLEARN_OK, bool)
print("Step 0 integrity check passed.")


DEVICE: cpu
Torch: 2.10.0+cpu
Step 0 integrity check passed.


In [4]:
# Step 1. Mount drive + resolve paths + unified run/output dirs
import os
from typing import Dict, List

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("[INFO] google.colab drive mount skipped:", repr(e))

# ===== experiment identity (EDIT THIS ONLY) =====
EXP_ID = f"gorV3"

# ===== NEW: fixed input/output roots =====
DATA_DIR = "/content/drive/MyDrive/scMILD_final/data"   # inputs live here
RUNS_DIR = "/content/drive/MyDrive/scMILD_final/runs"  # outputs live here

assert os.path.isdir(DATA_DIR), f"DATA_DIR not found: {DATA_DIR}"
os.makedirs(RUNS_DIR, exist_ok=True)

BASE_DIR = DATA_DIR
SPLITS = ["train", "val", "test"]

PATHS: Dict[str, Dict[str, str]] = {}
for sp in SPLITS:
    PATHS[sp] = {
        "z": os.path.join(BASE_DIR, f"z_{sp}.npy"),
        "obs": os.path.join(BASE_DIR, f"obs_{sp}.csv"),
        "bagmap": os.path.join(BASE_DIR, f"bagmap_{sp}_seed0.json"),
    }

# ===== NEW: run dir under RUNS_DIR (NOT under BASE_DIR) =====
RUN_DIR  = os.path.join(RUNS_DIR, EXP_ID)
OUT_DIR  = os.path.join(RUN_DIR, "eval_artifacts")
CKPT_DIR = os.path.join(RUN_DIR, "checkpoints")
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RUN_DIR :", RUN_DIR)
print("OUT_DIR :", OUT_DIR)
print("CKPT_DIR:", CKPT_DIR)

# Integrity test
for sp in SPLITS:
    for k in ["z", "obs", "bagmap"]:
        print(sp, k, PATHS[sp][k], "EXISTS=", os.path.exists(PATHS[sp][k]))
        assert os.path.exists(PATHS[sp][k]), f"Missing file: {PATHS[sp][k]}"
print("Step 1 integrity check passed.")

Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/scMILD_final/data
RUN_DIR : /content/drive/MyDrive/scMILD_final/runs/gorV3
OUT_DIR : /content/drive/MyDrive/scMILD_final/runs/gorV3/eval_artifacts
CKPT_DIR: /content/drive/MyDrive/scMILD_final/runs/gorV3/checkpoints
train z /content/drive/MyDrive/scMILD_final/data/z_train.npy EXISTS= True
train obs /content/drive/MyDrive/scMILD_final/data/obs_train.csv EXISTS= True
train bagmap /content/drive/MyDrive/scMILD_final/data/bagmap_train_seed0.json EXISTS= True
val z /content/drive/MyDrive/scMILD_final/data/z_val.npy EXISTS= True
val obs /content/drive/MyDrive/scMILD_final/data/obs_val.csv EXISTS= True
val bagmap /content/drive/MyDrive/scMILD_final/data/bagmap_val_seed0.json EXISTS= True
test z /content/drive/MyDrive/scMILD_final/data/z_test.npy EXISTS= True
test obs /content/drive/MyDrive/scMILD_final/data/obs_test.csv EXISTS= True
test bagmap /content/drive/MyDrive/scMILD_final/data/bagmap_test_seed0.json EXISTS= True
Step 1 integri

In [5]:

# Step 2. Load raw split data
RAW: Dict[str, Dict[str, Any]] = {}

for sp in SPLITS:
    z = np.load(PATHS[sp]["z"])
    obs = pd.read_csv(PATHS[sp]["obs"])
    with open(PATHS[sp]["bagmap"], "r") as f:
        bagmap = json.load(f)
    RAW[sp] = {"z": z, "obs": obs, "bagmap": bagmap}
    print(f"[{sp}] z.shape={z.shape}, obs.shape={obs.shape}, n_bags={len(bagmap)}")

# Integrity tests
for sp in SPLITS:
    z = RAW[sp]["z"]
    obs = RAW[sp]["obs"]
    bagmap = RAW[sp]["bagmap"]

    assert isinstance(z, np.ndarray) and z.ndim == 2, f"{sp}: z must be 2D np.ndarray"
    assert len(obs) == z.shape[0], f"{sp}: obs rows != z rows"
    assert isinstance(bagmap, dict) and len(bagmap) > 0, f"{sp}: bagmap invalid"
    assert np.issubdtype(z.dtype, np.number), f"{sp}: z not numeric"
    idx_sample = np.random.choice(z.shape[0], size=min(10000, z.shape[0]), replace=False)
    assert np.isfinite(z[idx_sample]).all(), f"{sp}: z contains NaN/Inf"
print("Step 2 integrity check passed.")


[train] z.shape=(644021, 32), obs.shape=(644021, 4), n_bags=198
[val] z.shape=(150216, 32), obs.shape=(150216, 4), n_bags=42
[test] z.shape=(120509, 32), obs.shape=(120509, 4), n_bags=42
Step 2 integrity check passed.


In [6]:

# Step 3. Standardize obs columns + normalize labels
def find_col(df: pd.DataFrame, candidates: List[str]) -> str:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    raise KeyError(f"None of columns found. candidates={candidates}, actual={list(df.columns)}")

def normalize_label(y_raw: str) -> str:
    s = str(y_raw).strip().lower()

    # normalize separators/spaces
    s2 = s.replace(" ", "").replace("-", "_")

    # control
    if ("control" in s2) or (s2 in ["healthy", "normal", "hc"]):
        return "control"

    # mild/moderate
    if ("mild" in s2) or ("moderate" in s2) or (s2 in ["mild_moderate", "mild/moderate", "mildmoderate"]):
        return "mild/moderate"

    # severe/critical
    if ("severe" in s2) or ("critical" in s2) or (s2 in ["severe_critical", "severe/critical", "severecritical"]):
        return "severe/critical"

    raise ValueError(f"Unknown label string: {y_raw}")
LABEL_TO_INT = {"control": 0, "mild/moderate": 1, "severe/critical": 2}
INT_TO_LABEL = {v: k for k, v in LABEL_TO_INT.items()}

STANDARD: Dict[str, pd.DataFrame] = {}
OBS_COLS_USED: Dict[str, Dict[str, str]] = {}

for sp in SPLITS:
    obs = RAW[sp]["obs"].copy()

    sample_col = find_col(obs, ["sampleID", "sample_id", "sample", "SampleID", "bag_id"])
    donor_col = find_col(obs, ["donor_id", "donor", "patient_id", "DonorID", "donorID"])
    y_col = find_col(obs, ["y", "label", "group", "condition"])

    obs["_sampleID"] = obs[sample_col].astype(str)
    obs["_donor_id"] = obs[donor_col].astype(str)
    obs["_y_raw"] = obs[y_col].astype(str)
    obs["_y_norm"] = obs["_y_raw"].map(normalize_label)
    obs["_y_int"] = obs["_y_norm"].map(LABEL_TO_INT).astype(int)

    STANDARD[sp] = obs
    OBS_COLS_USED[sp] = {"sample_col": sample_col, "donor_col": donor_col, "y_col": y_col}

    print(f"[{sp}] standardized columns: sample={sample_col}, donor={donor_col}, y={y_col}")
    print(obs[['_sampleID', '_donor_id', '_y_raw', '_y_norm', '_y_int']].head(2))

# Integrity tests
for sp in SPLITS:
    obs = STANDARD[sp]
    assert obs["_y_int"].isin([0,1,2]).all(), f"{sp}: invalid label mapping"
    assert obs["_sampleID"].notna().all() and obs["_donor_id"].notna().all()
    assert obs["_sampleID"].astype(str).str.len().gt(0).all()
print("Step 3 integrity check passed.")

for sp in SPLITS:
    obs = STANDARD[sp]
    nunique_by_sample = obs.groupby("_sampleID")["_y_int"].nunique()
    bad = nunique_by_sample[nunique_by_sample > 1]
    assert len(bad) == 0, f"{sp}: same sampleID has multiple labels. Examples: {bad.head().to_dict()}"
print("Step 3 sampleID-label consistency passed.")


[train] standardized columns: sample=sampleID, donor=donor_id, y=y
  _sampleID _donor_id           _y_raw          _y_norm  _y_int
0  S-S070-1    P-S070  severe/critical  severe/critical       2
1  S-S070-1    P-S070  severe/critical  severe/critical       2
[val] standardized columns: sample=sampleID, donor=donor_id, y=y
  _sampleID _donor_id         _y_raw        _y_norm  _y_int
0  S-M044-1    P-M044  mild/moderate  mild/moderate       1
1  S-M044-1    P-M044  mild/moderate  mild/moderate       1
[test] standardized columns: sample=sampleID, donor=donor_id, y=y
  _sampleID _donor_id           _y_raw          _y_norm  _y_int
0  S-S072-2    P-S072  severe/critical  severe/critical       2
1  S-S072-2    P-S072  severe/critical  severe/critical       2
Step 3 integrity check passed.
Step 3 sampleID-label consistency passed.


In [7]:
import json
from pprint import pprint

# RAW['train']['bagmap']이 이미 로드돼 있다고 가정
bm = RAW["train"]["bagmap"]

print(type(bm))
if isinstance(bm, dict):
    k0 = next(iter(bm.keys()))
    print("first key:", k0)
    print("value type:", type(bm[k0]))
    print("value head:", bm[k0] if not isinstance(bm[k0], list) else bm[k0][:10])
elif isinstance(bm, list):
    print("first item keys:", bm[0].keys())
    pprint(bm[0])

<class 'dict'>
first key: S-S070-1
value type: <class 'dict'>
value head: {'idx': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 20

In [8]:
# Step 4. Build bag-level records from bagmap + obs cross-check (CLEAN)

from typing import Any, Dict, List

# 4-0) bagmap: dict[bag_id] -> {"idx":[...], "donor_id":..., "y":..., "n_total":..., "n_cap":...}
def bagmap_to_idx(bm_raw: Dict[str, Any]) -> Dict[str, List[int]]:
    """Extract only idx list from your bagmap format."""
    assert isinstance(bm_raw, dict)
    out: Dict[str, List[int]] = {}
    for bag_id, rec in bm_raw.items():
        if not isinstance(rec, dict):
            raise TypeError(f"bagmap[{bag_id}] must be dict, got {type(rec)}")
        if "idx" not in rec:
            raise KeyError(f"bagmap[{bag_id}] missing 'idx'. keys={list(rec.keys())}")
        idxs = rec["idx"]
        if not isinstance(idxs, list) or len(idxs) == 0:
            raise ValueError(f"bagmap[{bag_id}] idx must be non-empty list, got type={type(idxs)} len={len(idxs) if isinstance(idxs,list) else 'NA'}")
        out[str(bag_id)] = [int(i) for i in idxs]
    return out

# 4-1) build bag_table from bagmap meta (authoritative), then cross-check with obs
BAGMAP_IDX: Dict[str, Dict[str, List[int]]] = {}
BAG_TABLE: Dict[str, pd.DataFrame] = {}
BAGS: Dict[str, List[Dict[str, Any]]] = {}

for sp in SPLITS:
    obs = STANDARD[sp].copy()
    bm_raw = RAW[sp]["bagmap"]            # dict: bag_id -> dict(meta+idx)
    zmat = RAW[sp]["z"]                   # np.ndarray [n_cells, d]
    n_cells = int(zmat.shape[0])

    # --- A) extract idx-only map ---
    bm_idx = bagmap_to_idx(bm_raw)
    BAGMAP_IDX[sp] = bm_idx

    # --- B) key alignment (bag_id must match obs _sampleID) ---
    obs_bags = set(obs["_sampleID"].astype(str).unique())
    bm_bags  = set(bm_idx.keys())

    missing_in_obs = bm_bags - obs_bags
    assert len(missing_in_obs) == 0, (
        f"[{sp}] bagmap keys not in obs sampleID. examples={list(sorted(missing_in_obs))[:10]}"
    )

    extra_in_obs = obs_bags - bm_bags
    if len(extra_in_obs) > 0:
        print(f"[WARN][{sp}] obs has sampleIDs not in bagmap: n={len(extra_in_obs)} show5={list(sorted(extra_in_obs))[:5]}")

    # --- C) index range validity (sample a few + fast global check) ---
    for bid in list(sorted(bm_idx.keys()))[:10]:
        idxs = bm_idx[bid]
        mi, ma = min(idxs), max(idxs)
        assert 0 <= mi and ma < n_cells, f"[{sp}] idx out of bounds for {bid}: min={mi}, max={ma}, n_cells={n_cells}"

    # Optionally do a faster global check (cheap)
    # (comment out if too slow, but usually fine)
    for bid, idxs in list(bm_idx.items())[:200]:  # sample first 200 bags
        mi, ma = min(idxs), max(idxs)
        if mi < 0 or ma >= n_cells:
            raise IndexError(f"[{sp}] idx out of bounds for {bid}: min={mi}, max={ma}, n_cells={n_cells}")

    # --- D) obs consistency: one label per sampleID ---
    nunique = obs.groupby("_sampleID")["_y_int"].nunique()
    bad = nunique[nunique > 1]
    assert len(bad) == 0, f"[{sp}] sampleID has multiple labels in obs. examples={bad.head().to_dict()}"

    # --- E) cross-check bagmap meta vs obs meta (donor/y) ---
    # obs meta per bag
    obs_meta = (
        obs[["_sampleID", "_donor_id", "_y_norm"]]
        .drop_duplicates("_sampleID")
        .set_index("_sampleID")
    )

    bm_meta_rows = []
    for bag_id, rec in bm_raw.items():
        bag_id = str(bag_id)
        bm_meta_rows.append({
            "_sampleID": bag_id,
            "bm_donor_id": str(rec.get("donor_id", "")),
            "bm_y_raw": str(rec.get("y", "")),
            "bm_y_norm": normalize_label(str(rec.get("y", ""))),
            "bm_n_total": int(rec.get("n_total", len(rec["idx"]))),
            "bm_n_cap": int(rec.get("n_cap", len(rec["idx"]))),
            "bm_len_idx": int(len(rec["idx"])),
        })
    bm_meta = pd.DataFrame(bm_meta_rows).set_index("_sampleID")

    joined = obs_meta.join(bm_meta, how="inner")
    assert len(joined) == len(bm_meta), f"[{sp}] obs_meta vs bm_meta join mismatch: obs_meta={len(obs_meta)}, bm_meta={len(bm_meta)}, joined={len(joined)}"

    bad_donor = joined[joined["_donor_id"] != joined["bm_donor_id"]]
    if len(bad_donor) > 0:
        raise ValueError(f"[{sp}] donor_id mismatch obs vs bagmap. examples:\n{bad_donor.head(5)}")

    bad_y = joined[joined["_y_norm"] != joined["bm_y_norm"]]
    if len(bad_y) > 0:
        raise ValueError(f"[{sp}] y mismatch obs vs bagmap. examples:\n{bad_y.head(5)}")

    # n_total sanity (optional strict)
    bad_n = joined[joined["bm_n_total"] != joined["bm_len_idx"]]
    if len(bad_n) > 0:
        # not necessarily fatal, but warn loudly
        print(f"[WARN][{sp}] n_total != len(idx) for some bags. examples:\n{bad_n.head(3)[['bm_n_total','bm_len_idx']]}")

    print(f"[{sp}] obs↔bagmap meta consistency OK.")

    # --- F) build bag-level table (authoritative from bagmap meta) ---
    bag_df = pd.DataFrame({
        "bag_id": joined.index.astype(str),
        "donor_id": joined["_donor_id"].astype(str).values,
        "y_norm": joined["_y_norm"].astype(str).values,
        "y_true_int": joined["bm_y_norm"].map(LABEL_TO_INT).astype(int).values,
        "n_cells": joined["bm_len_idx"].astype(int).values,
        "n_total_json": joined["bm_n_total"].astype(int).values,
        "n_cap_json": joined["bm_n_cap"].astype(int).values,
    })

    # Hard: bag_df bag_id == bagmap keys
    assert set(bag_df["bag_id"]) == bm_bags, f"[{sp}] bag_df keys != bagmap keys"
    BAG_TABLE[sp] = bag_df

    # --- G) build training/eval records ---
    records: List[Dict[str, Any]] = []
    for bag_id, rec in bm_raw.items():
        bag_id = str(bag_id)
        idx = np.asarray(rec["idx"], dtype=np.int64)
        if idx.ndim != 1:
            raise ValueError(f"[{sp}] {bag_id}: idx not 1D")
        if idx.min() < 0 or idx.max() >= n_cells:
            raise IndexError(f"[{sp}] {bag_id}: idx out of bounds [{idx.min()}, {idx.max()}] vs n_cells={n_cells}")

        # since we already enforced obs_meta == bm_meta, we can trust either
        y_norm = normalize_label(rec["y"])
        records.append({
            "split": sp,
            "bag_id": bag_id,
            "idx": idx,
            "n_cells": int(len(idx)),
            "donor_id": str(rec["donor_id"]),
            "y_norm": y_norm,
            "y_int": int(LABEL_TO_INT[y_norm]),
            "n_total_json": int(rec.get("n_total", len(idx))),
            "n_cap_json": int(rec.get("n_cap", len(idx))),
        })

    # integrity: unique bag_id + label range
    seen = set()
    for r in records:
        assert r["bag_id"] not in seen, f"[{sp}] duplicate bag_id {r['bag_id']}"
        seen.add(r["bag_id"])
        assert r["y_int"] in [0,1,2]
        assert isinstance(r["idx"], np.ndarray) and r["idx"].dtype == np.int64
        assert len(r["idx"]) == r["n_cells"]

    BAGS[sp] = records
    print(f"[{sp}] Step4 OK: bags={len(records)}, n_cells(mean)={bag_df['n_cells'].mean():.1f}")

print("Step 4 integrity check passed.")

[train] obs↔bagmap meta consistency OK.
[train] Step4 OK: bags=198, n_cells(mean)=3252.6
[val] obs↔bagmap meta consistency OK.
[val] Step4 OK: bags=42, n_cells(mean)=3576.6
[test] obs↔bagmap meta consistency OK.
[test] Step4 OK: bags=42, n_cells(mean)=2869.3
Step 4 integrity check passed.


In [9]:
# Step 5. Split integrity tables (donor leakage / class counts) + save metadata summaries

def bags_to_df(BAGS: Dict[str, List[Dict[str, Any]]]) -> pd.DataFrame:
    rows = []
    for sp, recs in BAGS.items():
        for r in recs:
            rows.append({
                "split": sp,
                "bag_id": r["bag_id"],
                # bag_id == sampleID in your data
                "sample_id": r["bag_id"],
                "donor_id": r["donor_id"],
                "y_true_str": r["y_norm"],
                "y_true_int": r["y_int"],
                "n_cells": r["n_cells"],
                "n_total_json": r.get("n_total_json", r["n_cells"]),
                "n_cap_json": r.get("n_cap_json", r["n_cells"]),
            })
    return pd.DataFrame(rows)

bags_all = bags_to_df(BAGS)

def assert_no_duplicate_across_splits(df: pd.DataFrame, key: str):
    dup = df.groupby(key)["split"].nunique()
    bad = dup[dup > 1]
    assert len(bad) == 0, f"[LEAK] {key} appears in multiple splits. examples={bad.index.tolist()[:10]}"
    print(f"[OK] no cross-split leakage by {key}")

# bag_id와 sample_id는 동일하므로 둘 다 체크하는 건 중복이지만, 남겨도 됨
assert_no_duplicate_across_splits(bags_all, "bag_id")

# donor leakage 체크
donor_sets = {sp: set(bags_all.loc[bags_all["split"] == sp, "donor_id"].astype(str)) for sp in SPLITS}
pairs = []
for i, a in enumerate(SPLITS):
    for b in SPLITS[i+1:]:
        inter = donor_sets[a] & donor_sets[b]
        pairs.append({"split_a": a, "split_b": b, "n_overlap": len(inter)})
donor_overlap_pairs = pd.DataFrame(pairs)
print(donor_overlap_pairs)

# class counts
class_count = (
    bags_all.groupby(["split", "y_true_int"])["bag_id"]
    .nunique()
    .reset_index(name="n_bags")
)
print(class_count)

# Save artifacts
bags_all.to_csv(os.path.join(RUN_DIR, "bags_all.csv"), index=False)
donor_overlap_pairs.to_csv(os.path.join(RUN_DIR, "donor_overlap_pairs.csv"), index=False)
class_count.to_csv(os.path.join(RUN_DIR, "bag_class_count.csv"), index=False)

print("Step 5 OK. Saved: bags_all.csv / donor_overlap_pairs.csv / bag_class_count.csv")

[OK] no cross-split leakage by bag_id
  split_a split_b  n_overlap
0   train     val          0
1   train    test          0
2     val    test          0
   split  y_true_int  n_bags
0   test           0       5
1   test           1      16
2   test           2      21
3  train           0      18
4  train           1      87
5  train           2      93
6    val           0       5
7    val           1      19
8    val           2      18
Step 5 OK. Saved: bags_all.csv / donor_overlap_pairs.csv / bag_class_count.csv


In [10]:

# Step 6. Dataset / DataLoader
@dataclass
class DataConfig:
    train_cell_cap: Optional[int] = 4096
    eval_cell_cap: Optional[int] = 4096    # set None for full-bag eval
    batch_size: int = 8
    num_workers: int = 0
    pin_memory: bool = True
    drop_last_train: bool = False

DATA_CFG = DataConfig()

class BagDataset(Dataset):
    def __init__(self, z_np: np.ndarray, bag_records: List[Dict[str, Any]], split: str):
        self.z_np = z_np.astype(np.float32, copy=False)
        self.records = bag_records
        self.split = split
        self.d = int(self.z_np.shape[1])

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        r = self.records[i]
        return {
            "bag_id": r["bag_id"],
            "sample_id": r.get("sample_id", r["bag_id"]),
            "donor_id": r["donor_id"],
            "y_int": int(r["y_int"]),
            "y_norm": r["y_norm"],
            "idx": r["idx"],   # np.int64 array
            "n_cells": int(r["n_cells"]),
        }

def make_collate_fn(z_np: np.ndarray, training: bool, cell_cap: Optional[int] = None):
    z_np = z_np.astype(np.float32, copy=False)

    def _collate(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        xs, lens, ys = [], [], []
        meta = {"bag_id": [], "sample_id": [], "donor_id": [], "y_norm": []}

        for item in batch:
            idx = item["idx"]
            if (cell_cap is not None) and (len(idx) > cell_cap):
                if training:
                    pick = np.random.choice(len(idx), size=cell_cap, replace=False)
                    idx_use = idx[pick]
                else:
                    idx_use = idx[:cell_cap]
            else:
                idx_use = idx

            x = torch.from_numpy(z_np[idx_use])  # [n_i, d]
            xs.append(x)
            lens.append(int(x.shape[0]))
            ys.append(int(item["y_int"]))
            for k in meta.keys():
                meta[k].append(item[k])

        B = len(xs)
        Nmax = max(lens)
        d = int(xs[0].shape[1])

        x_pad = torch.zeros(B, Nmax, d, dtype=torch.float32)
        mask = torch.zeros(B, Nmax, dtype=torch.bool)

        for i, x in enumerate(xs):
            n = int(x.shape[0])
            x_pad[i, :n] = x
            mask[i, :n] = True

        y = torch.tensor(ys, dtype=torch.long)
        lens_t = torch.tensor(lens, dtype=torch.long)

        return {"x": x_pad, "mask": mask, "y": y, "lens": lens_t, **meta}

    return _collate

train_ds = BagDataset(RAW["train"]["z"], BAGS["train"], "train")
val_ds   = BagDataset(RAW["val"]["z"],   BAGS["val"],   "val")
test_ds  = BagDataset(RAW["test"]["z"],  BAGS["test"],  "test")

train_loader = DataLoader(
    train_ds, batch_size=DATA_CFG.batch_size, shuffle=True,
    num_workers=DATA_CFG.num_workers, pin_memory=DATA_CFG.pin_memory,
    drop_last=DATA_CFG.drop_last_train,
    collate_fn=make_collate_fn(RAW["train"]["z"], training=True, cell_cap=DATA_CFG.train_cell_cap),
)
val_loader = DataLoader(
    val_ds, batch_size=DATA_CFG.batch_size, shuffle=False,
    num_workers=DATA_CFG.num_workers, pin_memory=DATA_CFG.pin_memory,
    collate_fn=make_collate_fn(RAW["val"]["z"], training=False, cell_cap=DATA_CFG.eval_cell_cap),
)
test_loader = DataLoader(
    test_ds, batch_size=DATA_CFG.batch_size, shuffle=False,
    num_workers=DATA_CFG.num_workers, pin_memory=DATA_CFG.pin_memory,
    collate_fn=make_collate_fn(RAW["test"]["z"], training=False, cell_cap=DATA_CFG.eval_cell_cap),
)

print("Datasets/Loaders ready.")
print("n_train_bags:", len(train_ds), "n_val_bags:", len(val_ds), "n_test_bags:", len(test_ds))

# Integrity tests
batch = next(iter(train_loader))
B, Nmax, d = batch["x"].shape
assert batch["mask"].shape == (B, Nmax)
assert batch["y"].shape == (B,)
assert (batch["lens"] <= Nmax).all()
assert torch.equal(batch["mask"].sum(dim=1).cpu(), batch["lens"])
assert d == RAW["train"]["z"].shape[1]
print("Sample batch x:", batch["x"].shape, "mask:", batch["mask"].shape, "y:", batch["y"].tolist())
print("Step 6 integrity check passed.")


Datasets/Loaders ready.
n_train_bags: 198 n_val_bags: 42 n_test_bags: 42
Sample batch x: torch.Size([8, 4096, 32]) mask: torch.Size([8, 4096]) y: [1, 2, 0, 2, 1, 2, 1, 1]
Step 6 integrity check passed.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [12]:

# Step 7. Model definition (GOR v2 structure = concat residual + retained prototype coordinates)
# NOTE: This is NOT pure d-dimensional GOR. It keeps the original effective structure:
#   z_prog = concat([r, a_keep]) with dimension (d_branch + K_proto).
# We modify TRAINING regularization to avoid beta collapse.

@dataclass
class ModelConfig:
    d_in: int = int(RAW["train"]["z"].shape[1])  # e.g., 32
    d_hidden: int = 64
    d_branch: int = 64
    attn_dim: int = 64
    K_proto: int = 12
    dropout: float = 0.1
    beta_init: float = 0.7
    sev_head_hidden: int = 64

MODEL_CFG = ModelConfig()

def safe_logit(p: float, eps: float = 1e-6) -> float:
    p = min(max(float(p), eps), 1.0 - eps)
    return math.log(p / (1.0 - p))

class AttnMILBranch(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, attn_dim, dropout=0.1):
        super().__init__()
        self.inst = nn.Sequential(
            nn.LayerNorm(d_in),
            nn.Linear(d_in, d_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_hidden, d_out), nn.GELU(), nn.Dropout(dropout),
        )
        self.q_proj = nn.Linear(d_out, attn_dim, bias=False)  # for orth penalty
        self.g_proj = nn.Linear(d_out, attn_dim, bias=False)
        self.v = nn.Parameter(torch.randn(attn_dim) * 0.02)
        self.post = nn.Sequential(
            nn.LayerNorm(d_out),
            nn.Linear(d_out, d_out), nn.GELU(), nn.Dropout(dropout),
        )

    def forward(self, x, mask):
        # x: [B,N,d], mask: [B,N]
        h = self.inst(x)  # [B,N,d_out]
        q = torch.tanh(self.q_proj(h))
        g = torch.sigmoid(self.g_proj(h))
        score = torch.einsum("bnh,h->bn", q * g, self.v) / math.sqrt(q.shape[-1])
        score = score.masked_fill(~mask, -1e9)
        a = torch.softmax(score, dim=1)
        z = torch.einsum("bn,bnd->bd", a, h)
        z = self.post(z)
        return z, a, score, h

class QRPrototypeDiseaseHead(nn.Module):
    def __init__(self, d_branch, K_proto):
        super().__init__()
        self.d_branch = d_branch
        self.K_proto = K_proto
        self.V_raw = nn.Parameter(torch.randn(d_branch, K_proto) * 0.02)
        self.bias = nn.Parameter(torch.zeros(K_proto))

    def get_W(self):
        Q, R = torch.linalg.qr(self.V_raw, mode='reduced')  # Q: [d, K]
        W = Q.transpose(0, 1)                               # W: [K, d]
        return W, Q, R

    def forward(self, z_d):
        W, Q, R = self.get_W()
        logits_each = z_d @ W.t() + self.bias   # [B,K]
        p_each = torch.sigmoid(logits_each)
        p_sick, idx = p_each.max(dim=1)
        return {
            "p_sick": p_sick,
            "logits_each": logits_each,
            "p_each": p_each,
            "proto_idx": idx,
            "W": W, "Q": Q, "R": R,
        }

class GatedOrthogonalResidual(nn.Module):
    '''
    z_s -> a = z_s W^T (K), z_parallel = aW, r = z_s - z_parallel
    beta in [0,1]^K
    a_keep = a * (1 - beta)
    z_prog = concat([r, a_keep])  # dimension d_branch + K
    '''
    def __init__(self, K_proto, beta_init=0.7):
        super().__init__()
        self.beta_logit = nn.Parameter(torch.full((K_proto,), safe_logit(beta_init), dtype=torch.float32))

    def forward(self, z_s, W):
        a = z_s @ W.t()                 # [B,K]
        z_parallel = a @ W              # [B,d]
        r = z_s - z_parallel            # [B,d]
        beta = torch.sigmoid(self.beta_logit)  # [K]
        a_keep = a * (1.0 - beta.unsqueeze(0))
        z_prog = torch.cat([r, a_keep], dim=-1)
        return {
            "z_prog": z_prog, "a": a, "a_keep": a_keep,
            "z_parallel": z_parallel, "r": r, "beta": beta,
        }

class SeverityHead(nn.Module):
    def __init__(self, d_in, hidden=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_in),
            nn.Linear(d_in, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, z_prog):
        logit = self.net(z_prog).squeeze(-1)
        p = torch.sigmoid(logit)
        return logit, p

class GORHierMIL(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.branch_d = AttnMILBranch(cfg.d_in, cfg.d_hidden, cfg.d_branch, cfg.attn_dim, cfg.dropout)
        self.branch_s = AttnMILBranch(cfg.d_in, cfg.d_hidden, cfg.d_branch, cfg.attn_dim, cfg.dropout)
        self.disease_head = QRPrototypeDiseaseHead(cfg.d_branch, cfg.K_proto)
        self.gor = GatedOrthogonalResidual(cfg.K_proto, cfg.beta_init)
        self.sev_head = SeverityHead(cfg.d_branch + cfg.K_proto, cfg.sev_head_hidden, cfg.dropout)

    def query_orth_penalty(self):
        W1 = self.branch_d.q_proj.weight
        W2 = self.branch_s.q_proj.weight
        prod = W1 @ W2.t()
        return prod.pow(2).mean()

    def forward(self, x, mask):
        z_d, a_d, score_d, h_d = self.branch_d(x, mask)
        z_s, a_s, score_s, h_s = self.branch_s(x, mask)

        disease = self.disease_head(z_d)
        gor_out = self.gor(z_s, disease["W"].detach())
        sev_logit, p_sev_cond = self.sev_head(gor_out["z_prog"])
        p_sick = disease["p_sick"]

        p_H = 1.0 - p_sick
        p_M = p_sick * (1.0 - p_sev_cond)
        p_S = p_sick * p_sev_cond
        p_3 = torch.stack([p_H, p_M, p_S], dim=-1)

        return {
            "z_d": z_d, "z_s": z_s,
            "attn_d": a_d, "attn_s": a_s,
            "score_d": score_d, "score_s": score_s,
            **disease, **gor_out,
            "sev_logit": sev_logit, "p_sev_cond": p_sev_cond,
            "p_H": p_H, "p_M": p_M, "p_S": p_S, "p_3": p_3,
            "query_orth_pen": self.query_orth_penalty(),
        }

model = GORHierMIL(MODEL_CFG).to(DEVICE)
print(model.__class__.__name__, "initialized on", DEVICE)

# Integrity tests (forward + orthonormality + probability sanity)
model.eval()
with torch.no_grad():
    b = next(iter(train_loader))
    x = b["x"].to(DEVICE)
    mask = b["mask"].to(DEVICE)
    out = model(x, mask)
    B = x.shape[0]
    d_branch = MODEL_CFG.d_branch
    K = MODEL_CFG.K_proto

    assert out["z_d"].shape == (B, d_branch)
    assert out["z_s"].shape == (B, d_branch)
    assert out["W"].shape == (K, d_branch)
    assert out["z_prog"].shape == (B, d_branch + K)
    assert out["p_3"].shape == (B, 3)
    p3_sum = out["p_3"].sum(dim=1)
    assert torch.allclose(p3_sum, torch.ones_like(p3_sum), atol=1e-5)
    assert torch.all(out["p_S"] <= out["p_sick"] + 1e-6)

    WWt = out["W"] @ out["W"].t()
    I = torch.eye(K, device=WWt.device)
    ortho_err = (WWt - I).abs().max().item()
    print("QR row-orth max abs err:", ortho_err)
    assert ortho_err < 1e-4

    beta0 = out["beta"].detach().cpu().numpy()
    print("Initial beta:", beta0)

print("Step 7 integrity check passed.")


GORHierMIL initialized on cpu
QR row-orth max abs err: 3.5762786865234375e-07
Initial beta: [0.7 0.7 0.7 0.7 0.7 0.7 0.7 0.7 0.7 0.7 0.7 0.7]
Step 7 integrity check passed.


In [13]:
# Step 7.1. Conditional beta gate (bag-conditional, program-wise)

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# prereq integrity
for _k in ["MODEL_CFG", "safe_logit"]:
    assert _k in globals(), f"Missing prereq: {_k}. Run Step 7 cell first."

class BetaGateMLP(nn.Module):
    """
    Input : [a | r | z_s]  -> logits_beta in R^K
    Output: beta = sigmoid(logits_beta) in (0,1)^K, computed per bag
    """
    def __init__(self, d_branch: int, K_proto: int, hidden: int = 64, dropout: float = 0.1, beta_init: float = 0.7):
        super().__init__()
        self.d_branch = int(d_branch)
        self.K_proto = int(K_proto)
        self.d_in = self.K_proto + 2 * self.d_branch

        self.net = nn.Sequential(
            nn.LayerNorm(self.d_in),
            nn.Linear(self.d_in, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout),
        )
        self.out = nn.Linear(hidden, self.K_proto)

        # init: start near constant beta_init (so you don’t destabilize training)
        nn.init.zeros_(self.out.weight)
        with torch.no_grad():
            self.out.bias.fill_(safe_logit(beta_init))

    def forward(self, a: torch.Tensor, r: torch.Tensor, z_s: torch.Tensor):
        # a: [B,K], r: [B,d], z_s: [B,d]
        x = torch.cat([a, r, z_s], dim=-1)  # [B, K+2d]
        logits = self.out(self.net(x))      # [B,K]
        beta = torch.sigmoid(logits)        # [B,K]
        return logits, beta

# ---- integrity tests ----
B = 4
d = int(MODEL_CFG.d_branch)
K = int(MODEL_CFG.K_proto)
gate = BetaGateMLP(d_branch=d, K_proto=K, hidden=MODEL_CFG.sev_head_hidden, dropout=MODEL_CFG.dropout, beta_init=MODEL_CFG.beta_init)

a = torch.randn(B, K)
r = torch.randn(B, d)
z_s = torch.randn(B, d)

logits, beta = gate(a, r, z_s)

assert logits.shape == (B, K)
assert beta.shape == (B, K)
assert torch.isfinite(beta).all()
assert (beta > 0).all() and (beta < 1).all()

# init sanity: mean beta near beta_init
beta_mean = beta.mean().item()
print("[Step 7.1] beta mean at init:", beta_mean)
print("Step 7.1 integrity check passed.")

# Step 7.2. Conditional GOR (uses anchored W from disease head)

class ConditionalGatedOrthogonalResidual(nn.Module):
    """
    z_s -> a = z_s W^T (B,K)
         -> z_parallel = a W (B,d)
         -> r = z_s - z_parallel (B,d)
    beta(bag) = sigmoid(MLP([a|r|z_s])) (B,K)
    a_keep = a * (1 - beta)
    z_prog = concat([r, a_keep]) (B, d+K)
    """
    def __init__(self, d_branch: int, K_proto: int, hidden: int = 64, dropout: float = 0.1, beta_init: float = 0.7):
        super().__init__()
        self.d_branch = int(d_branch)
        self.K_proto = int(K_proto)
        self.gate = BetaGateMLP(d_branch=d_branch, K_proto=K_proto, hidden=hidden, dropout=dropout, beta_init=beta_init)

    def forward(self, z_s: torch.Tensor, W: torch.Tensor):
        # z_s: [B,d], W: [K,d]
        a = z_s @ W.t()            # [B,K]
        z_parallel = a @ W         # [B,d]
        r = z_s - z_parallel       # [B,d]

        beta_logit, beta = self.gate(a, r, z_s)  # [B,K]
        a_keep = a * (1.0 - beta)                # [B,K]
        z_prog = torch.cat([r, a_keep], dim=-1)  # [B,d+K]

        # keep also a global summary beta for legacy logging if you want
        beta_global = beta.mean(dim=0)           # [K]

        return {
            "z_prog": z_prog,
            "a": a, "a_keep": a_keep,
            "z_parallel": z_parallel, "r": r,
            "beta": beta,                 # NOTE: now [B,K]
            "beta_global": beta_global,   # [K] optional
            "beta_logit": beta_logit,     # [B,K]
        }

# ---- integrity tests ----
B = 3
d = int(MODEL_CFG.d_branch)
K = int(MODEL_CFG.K_proto)
gor_c = ConditionalGatedOrthogonalResidual(d_branch=d, K_proto=K, hidden=MODEL_CFG.sev_head_hidden,
                                          dropout=MODEL_CFG.dropout, beta_init=MODEL_CFG.beta_init)

z_s = torch.randn(B, d, requires_grad=True)
# make an orthonormal-ish W for test
W = torch.randn(K, d)
W = torch.linalg.qr(W.t(), mode="reduced").Q.t()  # [K,d] row-orthonormal

out = gor_c(z_s, W)

assert out["z_prog"].shape == (B, d + K)
assert out["beta"].shape == (B, K)
assert torch.isfinite(out["z_prog"]).all()
assert (out["beta"] > 0).all() and (out["beta"] < 1).all()

# gradient sanity: beta depends on z_s (should not be disconnected)
loss = out["beta"].sum() + out["z_prog"].sum()
loss.backward()
assert z_s.grad is not None and torch.isfinite(z_s.grad).all()
print("[Step 7.2] grad|z_s| mean:", z_s.grad.abs().mean().item())
print("Step 7.2 integrity check passed.")

# Step 7.3. Patch existing model.gor only (leave all other code untouched)

for _k in ["model", "train_loader", "DEVICE"]:
    assert _k in globals(), f"Missing prereq: {_k}. Run Step 7 + Step 6 first."

# swap gor
d = int(MODEL_CFG.d_branch)
K = int(MODEL_CFG.K_proto)
model.gor = ConditionalGatedOrthogonalResidual(
    d_branch=d, K_proto=K,
    hidden=MODEL_CFG.sev_head_hidden,
    dropout=MODEL_CFG.dropout,
    beta_init=MODEL_CFG.beta_init
).to(DEVICE)

# forward integrity test (same as your Step 7 test, but beta is now [B,K])
model.eval()
with torch.no_grad():
    b = next(iter(train_loader))
    x = b["x"].to(DEVICE)
    mask = b["mask"].to(DEVICE)
    out = model(x, mask)

    B = x.shape[0]
    assert out["z_d"].shape == (B, d)
    assert out["z_s"].shape == (B, d)
    assert out["W"].shape == (K, d)
    assert out["z_prog"].shape == (B, d + K)
    assert out["p_3"].shape == (B, 3)

    # probabilities sanity
    p3_sum = out["p_3"].sum(dim=1)
    assert torch.allclose(p3_sum, torch.ones_like(p3_sum), atol=1e-5)
    assert torch.all(out["p_S"] <= out["p_sick"] + 1e-6)

    # beta sanity
    beta = out["beta"]  # [B,K]
    assert beta.shape == (B, K)
    assert torch.isfinite(beta).all()
    print("[Step 7.3] beta mean/std/min/max:",
          beta.mean().item(), beta.std(unbiased=False).item(), beta.min().item(), beta.max().item())

print("Step 7.3 integrity check passed.")

[Step 7.1] beta mean at init: 0.7000000476837158
Step 7.1 integrity check passed.
[Step 7.2] grad|z_s| mean: 0.8342689871788025
Step 7.2 integrity check passed.
[Step 7.3] beta mean/std/min/max: 0.7000000476837158 5.960464477539063e-08 0.699999988079071 0.699999988079071
Step 7.3 integrity check passed.


In [14]:
# Step 8. Losses + anti-collapse regularization + stage control

from dataclasses import dataclass
from typing import Dict, Tuple, Optional
import numpy as np
import torch
import torch.nn.functional as F

@dataclass
class TrainConfig:
    lr: float = 1e-3
    weight_decay: float = 1e-4
    max_epochs: int = 60
    patience: int = 10
    grad_clip: float = 2.0

    # regularization
    lambda_query_orth: float = 1e-3

    # beta regularization (anti-collapse)
    lambda_beta_center: float = 0.0
    beta_target_mean: float = 0.70

    lambda_beta_spread: float = 3e-2
    lambda_beta_range: float = 3e-2
    beta_target_std: float = 0.08
    beta_target_range: float = 0.12

    # schedule for regs (only meaningful in stage=2)
    reg_warmup_epochs: int = 0
    reg_ramp_epochs: int = 3

TRAIN_CFG = TrainConfig()

def get_reg_scale(epoch: int, warmup: int, ramp: int) -> float:
    if epoch < warmup:
        return 0.0
    if ramp <= 0:
        return 1.0
    t = (epoch - warmup + 1) / ramp
    return float(min(1.0, max(0.0, t)))

def compute_probs_from_logits(logit_sick: torch.Tensor, logit_sev: torch.Tensor):
    """
    logit_sick: scalar tensor
    logit_sev : scalar tensor (interpreted as P(severe | sick))
    """
    p_sick = torch.sigmoid(logit_sick)
    p_sev_cond = torch.sigmoid(logit_sev)

    p_H = 1.0 - p_sick
    p_M = p_sick * (1.0 - p_sev_cond)
    p_S = p_sick * p_sev_cond
    return p_H, p_M, p_S, p_sick, p_sev_cond

def compute_losses(out: Dict[str, torch.Tensor], y: torch.Tensor, epoch: int, cfg: TrainConfig, stage: int = 2):
    """
    y: shape [B], values in {0,1,2}
    stage=1: disease-only (loss_sick만)
    stage=2: full (loss_sick + loss_sev + regs)
    """
    device = y.device
    y_int = y.long()

    # sick target: y>0
    y_sick = (y_int > 0).float()
    loss_sick = F.binary_cross_entropy(out["p_sick"], y_sick)

    # severe|sick target: y==2, mask for sick only
    y_sev = (y_int == 2).float()
    sick_mask = (y_int > 0).float()
    # out["p_sev_cond"] should exist; if you store logit use BCEWithLogits instead
    loss_sev_raw = F.binary_cross_entropy(out["p_sev_cond"], y_sev, reduction="none")
    loss_sev = (loss_sev_raw * sick_mask).sum() / (sick_mask.sum().clamp_min(1.0))

    # beta stats
    beta = out["beta"]
    beta_mean = beta.mean()
    beta_std = beta.std(unbiased=False)
    beta_range = beta.max() - beta.min()

    # stage 1: ONLY disease
    if stage == 1:
        total = loss_sick
        z = torch.zeros((), device=device)
        log = {
            "loss_total": total.detach(),
            "loss_sick": loss_sick.detach(),
            "loss_sev": z.detach(),
            "loss_qorth": z.detach(),
            "loss_beta_center": z.detach(),
            "loss_beta_spread": z.detach(),
            "loss_beta_range": z.detach(),
            "reg_scale": torch.tensor(0.0, device=device).detach(),
            "beta_mean": beta_mean.detach(),
            "beta_std": beta_std.detach(),
            "beta_range": beta_range.detach(),
        }
        return total, log

    # stage 2: regs on
    reg_scale = get_reg_scale(epoch, cfg.reg_warmup_epochs, cfg.reg_ramp_epochs)

    loss_qorth = out.get("query_orth_pen", torch.zeros((), device=device)) * (cfg.lambda_query_orth * reg_scale)
    loss_beta_center = (beta_mean - cfg.beta_target_mean).pow(2) * (cfg.lambda_beta_center * reg_scale)

    tstd = torch.tensor(cfg.beta_target_std, device=device, dtype=beta.dtype)
    trng = torch.tensor(cfg.beta_target_range, device=device, dtype=beta.dtype)
    loss_beta_spread = torch.relu(tstd - beta_std).pow(2) * (cfg.lambda_beta_spread * reg_scale)
    loss_beta_range  = torch.relu(trng - beta_range).pow(2) * (cfg.lambda_beta_range  * reg_scale)

    total = loss_sick + loss_sev + loss_qorth + loss_beta_center + loss_beta_spread + loss_beta_range

    log = {
        "loss_total": total.detach(),
        "loss_sick": loss_sick.detach(),
        "loss_sev": loss_sev.detach(),
        "loss_qorth": loss_qorth.detach(),
        "loss_beta_center": loss_beta_center.detach(),
        "loss_beta_spread": loss_beta_spread.detach(),
        "loss_beta_range": loss_beta_range.detach(),
        "reg_scale": torch.tensor(reg_scale, device=device).detach(),
        "beta_mean": beta_mean.detach(),
        "beta_std": beta_std.detach(),
        "beta_range": beta_range.detach(),
    }
    return total, log

In [17]:
# Step 8.1. Patch compute_losses to accept beta shape [K] or [B,K]

for _k in ["TrainConfig", "TRAIN_CFG"]:
    assert _k in globals(), f"Missing prereq: {_k}. Run Step 8 first."

def _beta_stats(beta: torch.Tensor):
    """
    Returns:
      beta_mean_global, beta_std_global, beta_range_global
      beta_bag_std_mean (mean over K of std across bags) if beta is [B,K], else 0
    """
    if beta.ndim == 1:
        mean = beta.mean()
        std = beta.std(unbiased=False)
        rng = beta.max() - beta.min()
        bag_std_mean = beta.new_tensor(0.0)
        return mean, std, rng, bag_std_mean

    if beta.ndim == 2:
        # global dispersion across all entries
        mean = beta.mean()
        std = beta.std(unbiased=False)
        rng = beta.max() - beta.min()
        # bag-conditional dispersion: per-program std across bags
        bag_std = beta.std(dim=0, unbiased=False)      # [K]
        bag_std_mean = bag_std.mean()                  # scalar
        return mean, std, rng, bag_std_mean

    raise ValueError(f"beta must be 1D or 2D, got shape={tuple(beta.shape)}")

# Optional: add to TRAIN_CFG dynamically (keeps old config working)
if not hasattr(TRAIN_CFG, "lambda_beta_bagvar"):
    TRAIN_CFG.lambda_beta_bagvar = 0.0          # start OFF
    TRAIN_CFG.beta_target_bag_std = 0.05        # small target; tune later

def compute_losses(out: Dict[str, torch.Tensor], y: torch.Tensor, epoch: int, cfg: TrainConfig, stage: int = 2):
    device = y.device
    y_int = y.long()

    # sick target
    y_sick = (y_int > 0).float()
    loss_sick = F.binary_cross_entropy(out["p_sick"], y_sick)

    # severe|sick
    y_sev = (y_int == 2).float()
    sick_mask = (y_int > 0).float()
    loss_sev_raw = F.binary_cross_entropy(out["p_sev_cond"], y_sev, reduction="none")
    loss_sev = (loss_sev_raw * sick_mask).sum() / (sick_mask.sum().clamp_min(1.0))

    beta = out["beta"]
    beta_mean, beta_std, beta_range, beta_bag_std_mean = _beta_stats(beta)

    # stage 1
    if stage == 1:
        total = loss_sick
        z = torch.zeros((), device=device)
        log = {
            "loss_total": total.detach(),
            "loss_sick": loss_sick.detach(),
            "loss_sev": z.detach(),
            "loss_qorth": z.detach(),
            "loss_beta_center": z.detach(),
            "loss_beta_spread": z.detach(),
            "loss_beta_range": z.detach(),
            "loss_beta_bagvar": z.detach(),
            "reg_scale": torch.tensor(0.0, device=device).detach(),
            "beta_mean": beta_mean.detach(),
            "beta_std": beta_std.detach(),
            "beta_range": beta_range.detach(),
            "beta_bag_std_mean": beta_bag_std_mean.detach(),
        }
        return total, log

    # stage 2
    reg_scale = get_reg_scale(epoch, cfg.reg_warmup_epochs, cfg.reg_ramp_epochs)

    loss_qorth = out.get("query_orth_pen", torch.zeros((), device=device)) * (cfg.lambda_query_orth * reg_scale)

    loss_beta_center = (beta_mean - cfg.beta_target_mean).pow(2) * (cfg.lambda_beta_center * reg_scale)

    tstd = torch.tensor(cfg.beta_target_std, device=device, dtype=beta.dtype)
    trng = torch.tensor(cfg.beta_target_range, device=device, dtype=beta.dtype)
    loss_beta_spread = torch.relu(tstd - beta_std).pow(2) * (cfg.lambda_beta_spread * reg_scale)
    loss_beta_range  = torch.relu(trng - beta_range).pow(2) * (cfg.lambda_beta_range  * reg_scale)

    # NEW (optional): enforce bag-conditional variability (prevents "same beta for every bag")
    tbag = torch.tensor(getattr(cfg, "beta_target_bag_std", 0.0), device=device, dtype=beta.dtype)
    lam_bag = float(getattr(cfg, "lambda_beta_bagvar", 0.0))
    loss_beta_bagvar = torch.relu(tbag - beta_bag_std_mean).pow(2) * (lam_bag * reg_scale)

    total = loss_sick + loss_sev + loss_qorth + loss_beta_center + loss_beta_spread + loss_beta_range + loss_beta_bagvar

    log = {
        "loss_total": total.detach(),
        "loss_sick": loss_sick.detach(),
        "loss_sev": loss_sev.detach(),
        "loss_qorth": loss_qorth.detach(),
        "loss_beta_center": loss_beta_center.detach(),
        "loss_beta_spread": loss_beta_spread.detach(),
        "loss_beta_range": loss_beta_range.detach(),
        "loss_beta_bagvar": loss_beta_bagvar.detach(),
        "reg_scale": torch.tensor(reg_scale, device=device).detach(),
        "beta_mean": beta_mean.detach(),
        "beta_std": beta_std.detach(),
        "beta_range": beta_range.detach(),
        "beta_bag_std_mean": beta_bag_std_mean.detach(),
    }
    return total, log

print("Step 8.1 compute_losses patch applied.")

# Step 8.1 integrity test

model.train()
b = next(iter(train_loader))
b = to_device_batch(b, DEVICE)
out = model(b["x"], b["mask"])

loss, log = compute_losses(out, b["y"], epoch=0, cfg=TRAIN_CFG, stage=2)
assert torch.isfinite(loss).all()
for k,v in log.items():
    assert torch.isfinite(v).all(), f"log {k} is NaN/Inf"

loss.backward()  # gradient flow sanity
print("[Step 8.1] loss:", float(loss.detach().cpu()))
print("Step 8.1 integrity check passed.")

Step 8.1 compute_losses patch applied.
[Step 8.1] loss: 1.1494050025939941
Step 8.1 integrity check passed.


In [16]:
# Step 9. Train/Eval utilities + metrics + exporters

import json
import numpy as np
import pandas as pd
from typing import Dict, Any, List, Tuple, Optional

import torch
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

def collect_basic_metrics(y_true: np.ndarray, p3: np.ndarray) -> dict:
    """
    y_true: shape [N], values in {0,1,2}
    p3:     shape [N,3], probabilities for [H,M,S] (or logits already softmaxed)
    """
    y_true = np.asarray(y_true).astype(int)
    p3 = np.asarray(p3)
    assert p3.ndim == 2 and p3.shape[1] == 3, f"p3 shape must be [N,3], got {p3.shape}"

    y_pred = p3.argmax(axis=1)

    out = {
        "n": int(len(y_true)),
        "acc_3class": float(accuracy_score(y_true, y_pred)),
        "bacc_3class": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1_3class": float(f1_score(y_true, y_pred, average="macro")),
        "confusion_3class": confusion_matrix(y_true, y_pred, labels=[0,1,2]).tolist(),
    }

    # sick vs control
    y_sick = (y_true > 0).astype(int)
    p_sick = 1.0 - p3[:, 0]      # p(H)=control, so sick = 1 - p(H)
    out["auc_sick"] = float(roc_auc_score(y_sick, p_sick))
    out["aupr_sick"] = float(average_precision_score(y_sick, p_sick))

    # severe vs mild among sick
    m = (y_true > 0)
    y_sev = (y_true[m] == 2).astype(int)
    # conditional severe probability among sick: p(S)/(p(M)+p(S))
    denom = (p3[m,1] + p3[m,2])
    p_sev_cond = np.divide(p3[m,2], denom, out=np.zeros_like(p3[m,2]), where=(denom > 0))
    out["auc_sev_cond"] = float(roc_auc_score(y_sev, p_sev_cond))
    out["aupr_sev_cond"] = float(average_precision_score(y_sev, p_sev_cond))

    # severe vs rest
    y_sev_rest = (y_true == 2).astype(int)
    out["auc_sev_rest"] = float(roc_auc_score(y_sev_rest, p3[:,2]))
    out["aupr_sev_rest"] = float(average_precision_score(y_sev_rest, p3[:,2]))

    return out

def to_device_batch(batch: dict, device: str) -> dict:
    """
    Moves tensor values in a batch dict to device.
    Keeps non-tensors (lists/strings) as-is.
    """
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device, non_blocking=True)
        else:
            out[k] = v
    return out
def _to_float(x):
    if isinstance(x, torch.Tensor):
        return float(x.detach().cpu())
    return float(x)

def compute_metrics_from_df(df: pd.DataFrame) -> Dict[str, float]:
    y = df["y_true_int"].to_numpy()
    yhat = df["y_pred_int"].to_numpy()

    out = {
        "n": int(len(df)),
        "acc_3class": float(accuracy_score(y, yhat)),
        "bacc_3class": float(balanced_accuracy_score(y, yhat)),
        "macro_f1_3class": float(f1_score(y, yhat, average="macro")),
        "confusion_3class": confusion_matrix(y, yhat, labels=[0,1,2]).tolist(),
    }

    # sick vs control
    y_sick = (y > 0).astype(int)
    out["auc_sick"] = float(roc_auc_score(y_sick, df["p_sick"].to_numpy()))
    out["aupr_sick"] = float(average_precision_score(y_sick, df["p_sick"].to_numpy()))

    # severe vs mild among sick only (conditional)
    m = (y > 0)
    y_sev = (y[m] == 2).astype(int)
    out["auc_sev_cond"] = float(roc_auc_score(y_sev, df.loc[m, "p_sev_cond"].to_numpy()))
    out["aupr_sev_cond"] = float(average_precision_score(y_sev, df.loc[m, "p_sev_cond"].to_numpy()))

    # severe vs rest
    y_sev_rest = (y == 2).astype(int)
    out["auc_sev_rest"] = float(roc_auc_score(y_sev_rest, df["p_S"].to_numpy()))
    out["aupr_sev_rest"] = float(average_precision_score(y_sev_rest, df["p_S"].to_numpy()))
    return out

@torch.no_grad()
def export_split_predictions_from_eval(model, loader, save_path: str, device: str):
    logs, metrics, df = eval_one_epoch(model, loader, epoch=0, cfg=TRAIN_CFG, device=device, stage=2)
    df.to_csv(save_path, index=False)
    print("saved", save_path)
    return df, metrics

def train_one_epoch(model, loader, optimizer, epoch: int, cfg: TrainConfig, stage: int = 2, device=DEVICE):
    model.train()
    logs, y_true_all, p3_all = [], [], []

    for batch in loader:
        batch = to_device_batch(batch, device)
        out = model(batch["x"], batch["mask"])
        loss, log = compute_losses(out, batch["y"], epoch=epoch, cfg=cfg, stage=stage)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if cfg.grad_clip is not None and cfg.grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()

        logs.append({k: float(v.detach().cpu()) for k, v in log.items()})
        y_true_all.append(batch["y"].detach().cpu().numpy())
        p3_all.append(out["p_3"].detach().cpu().numpy())

    y_true = np.concatenate(y_true_all)
    p3 = np.concatenate(p3_all)
    metrics = collect_basic_metrics(y_true, p3)
    mean_logs = {k: float(np.mean([d[k] for d in logs])) for k in logs[0].keys()} if logs else {}
    return mean_logs, metrics


@torch.no_grad()
def eval_one_epoch(model, loader, epoch: int, cfg: TrainConfig, stage: int = 2, device=DEVICE):
    model.eval()
    logs, y_true_all, p3_all = [], [], []

    for batch in loader:
        batch = to_device_batch(batch, device)
        out = model(batch["x"], batch["mask"])
        loss, log = compute_losses(out, batch["y"], epoch=epoch, cfg=cfg, stage=stage)

        logs.append({k: float(v.detach().cpu()) for k, v in log.items()})
        y_true_all.append(batch["y"].detach().cpu().numpy())
        p3_all.append(out["p_3"].detach().cpu().numpy())

    y_true = np.concatenate(y_true_all)
    p3 = np.concatenate(p3_all)
    metrics = collect_basic_metrics(y_true, p3)
    mean_logs = {k: float(np.mean([d[k] for d in logs])) for k in logs[0].keys()} if logs else {}
    return mean_logs, metrics

In [18]:
# Step 10. Two-stage training (Stage1 disease-only -> Stage2 severity) + best/last saving

seed_everything(42)
model = GORHierMIL(MODEL_CFG).to(DEVICE)

model.gor = ConditionalGatedOrthogonalResidual(
    d_branch=int(MODEL_CFG.d_branch),
    K_proto=int(MODEL_CFG.K_proto),
    hidden=MODEL_CFG.sev_head_hidden,
    dropout=MODEL_CFG.dropout,
    beta_init=MODEL_CFG.beta_init
).to(DEVICE)

model.eval()
b = next(iter(train_loader))
b = to_device_batch(b, DEVICE)
with torch.no_grad():
    out = model(b["x"], b["mask"])
print("beta shape:", tuple(out["beta"].shape))   # 기대: (B, K)
print("beta mean/std:", out["beta"].mean().item(), out["beta"].std(unbiased=False).item())

import numpy as np

# selperf: metrics only
# seldiv : metrics + tiny beta diversity bonus
SEL_TAG = "seldiv"  # or "selperf"

def score_for_model_selection(metrics: dict, logs: dict, cfg=TRAIN_CFG) -> float:
    # primary predictive terms
    auc_sev_cond = float(metrics.get("auc_sev_cond", 0.0))
    bacc3 = float(metrics.get("bacc_3class", 0.0))
    auc_sick = float(metrics.get("auc_sick", 0.0))

    s = 0.55 * auc_sev_cond + 0.30 * bacc3 + 0.15 * auc_sick

    # optional diversity bonus
    if SEL_TAG == "seldiv":
        beta_std = float(logs.get("beta_std", 0.0))
        beta_range = float(logs.get("beta_range", 0.0))

        std_bonus = np.clip(beta_std / max(cfg.beta_target_std, 1e-8), 0.0, 1.0)
        rng_bonus = np.clip(beta_range / max(cfg.beta_target_range, 1e-8), 0.0, 1.0)

        s += 0.02 * std_bonus + 0.03 * rng_bonus

    return float(s)

SELECTION_FORMULA_STR = (
    "0.55*auc_sev_cond + 0.30*bacc_3class + 0.15*auc_sick"
    + (" + 0.02*clip(beta_std/target_std) + 0.03*clip(beta_range/target_range)" if SEL_TAG=="seldiv" else "")
)

def set_requires_grad(m: nn.Module, flag: bool):
    for p in m.parameters():
        p.requires_grad = flag

def make_opt(m: nn.Module):
    return torch.optim.AdamW(filter(lambda p: p.requires_grad, m.parameters()),
                             lr=TRAIN_CFG.lr, weight_decay=TRAIN_CFG.weight_decay)

STAGE1_EPOCHS = 10  # 5~10 추천

# -------------------------
# Stage 1: disease-only (anchor W)
# -------------------------
set_requires_grad(model.branch_d, True)
set_requires_grad(model.disease_head, True)

set_requires_grad(model.branch_s, False)
set_requires_grad(model.gor, False)
set_requires_grad(model.sev_head, False)

optimizer = make_opt(model)
print("[Stage 1] trainable params:", sum(p.requires_grad for p in model.parameters()), "/", sum(1 for _ in model.parameters()))

for epoch in range(STAGE1_EPOCHS):
    tr_logs, tr_metrics = train_one_epoch(model, train_loader, optimizer, epoch=epoch, cfg=TRAIN_CFG, stage=1, device=DEVICE)
    va_logs, va_metrics = eval_one_epoch(model, val_loader, epoch=epoch, cfg=TRAIN_CFG, stage=1, device=DEVICE)
    print(f"[Stage1 {epoch:03d}] val auc_sick={va_metrics.get('auc_sick', np.nan):.4f} val bacc3={va_metrics.get('bacc_3class', np.nan):.4f}")

# W drift check baseline (optional, requires disease_head.get_W())
W0 = None
if hasattr(model.disease_head, "get_W"):
    W0 = model.disease_head.get_W()[0].detach().cpu().numpy()

# -------------------------
# Stage 2: freeze disease anchor, train severity/progression
# -------------------------
set_requires_grad(model.branch_d, False)
set_requires_grad(model.disease_head, False)

set_requires_grad(model.branch_s, True)
set_requires_grad(model.gor, True)
set_requires_grad(model.sev_head, True)

optimizer = make_opt(model)
print("[Stage 2] trainable params:", sum(p.requires_grad for p in model.parameters()), "/", sum(1 for _ in model.parameters()))

history: List[Dict[str, Any]] = []
best = {"epoch": -1, "score": -np.inf, "state_dict": None, "val_metrics": None, "val_logs": None}
patience_counter = 0

for epoch2 in range(TRAIN_CFG.max_epochs):
    tr_logs, tr_metrics = train_one_epoch(model, train_loader, optimizer, epoch=epoch2, cfg=TRAIN_CFG, stage=2, device=DEVICE)
    va_logs, va_metrics = eval_one_epoch(model, val_loader, epoch=epoch2, cfg=TRAIN_CFG, stage=2, device=DEVICE)

    # selection score (keep your existing score_for_model_selection + SELECTION_FORMULA_STR)
    cur_score = score_for_model_selection(va_metrics, va_logs)

    history.append({
        "stage": 2,
        "epoch2": int(epoch2),
        "train": {**tr_logs, **tr_metrics},
        "val": {**va_logs, **va_metrics},
        "selection_score": float(cur_score),
        "val_beta_std": float(va_logs.get("beta_std", np.nan)),
        "val_beta_range": float(va_logs.get("beta_range", np.nan)),
        "val_reg_scale": float(va_logs.get("reg_scale", np.nan)),
    })

    print(
        f"[Stage2 {epoch2:03d}] "
        f"val auc_sev_cond={va_metrics.get('auc_sev_cond', np.nan):.4f} "
        f"val auc_sick={va_metrics.get('auc_sick', np.nan):.4f} "
        f"score={cur_score:.4f} "
        f"reg_scale={va_logs.get('reg_scale', np.nan):.2f} "
        f"beta_std={va_logs.get('beta_std', np.nan):.4f} "
        f"beta_range={va_logs.get('beta_range', np.nan):.4f}"
    )

    if cur_score > best["score"]:
        best.update({
            "epoch": int(epoch2),
            "score": float(cur_score),
            "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            "val_metrics": va_metrics,
            "val_logs": va_logs,
        })
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= TRAIN_CFG.patience:
        print("Early stopping at stage2 epoch", epoch2)
        break

# W drift check (should be ~0 if frozen)
if W0 is not None:
    W1 = model.disease_head.get_W()[0].detach().cpu().numpy()
    print("W drift (fro):", float(np.linalg.norm(W1 - W0)))

# Save LAST (stage2 end state)
last_ckpt_path = os.path.join(CKPT_DIR, "last_ckpt.pt")
torch.save({"epoch2": int(epoch2),
            "model_state_dict": {k: v.detach().cpu() for k,v in model.state_dict().items()},
            "model_cfg": asdict(MODEL_CFG),
            "train_cfg": asdict(TRAIN_CFG),
            "exp_id": EXP_ID,
            "selection_formula": SELECTION_FORMULA_STR}, last_ckpt_path)
print("saved", last_ckpt_path)

# Save BEST
assert best["state_dict"] is not None
model.load_state_dict(best["state_dict"])

best_ckpt_path = os.path.join(CKPT_DIR, "best_ckpt.pt")
torch.save({"epoch2": int(best["epoch"]),
            "best_score": float(best["score"]),
            "model_state_dict": {k: v.detach().cpu() for k,v in model.state_dict().items()},
            "model_cfg": asdict(MODEL_CFG),
            "train_cfg": asdict(TRAIN_CFG),
            "exp_id": EXP_ID,
            "selection_formula": SELECTION_FORMULA_STR,
            "best_val_metrics": best["val_metrics"],
            "best_val_logs": best["val_logs"]}, best_ckpt_path)
print("saved", best_ckpt_path)

# Export predictions: use your existing exporter cell or write a simple one later.
with open(os.path.join(RUN_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=2)
print("saved", os.path.join(RUN_DIR, "history.json"))

beta shape: (8, 12)
beta mean/std: 0.7000000476837158 5.960464477539063e-08
[Stage 1] trainable params: 15 / 42


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[Stage1 000] val auc_sick=0.7784 val bacc3=0.3509


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[Stage1 001] val auc_sick=0.9351 val bacc3=0.4667


...

[Stage2 013] val auc_sev_cond=0.8480 val auc_sick=0.9297 score=0.8876 reg_scale=1.00 beta_std=0.3721 beta_range=0.9936


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[Stage2 014] val auc_sev_cond=0.8596 val auc_sick=0.9297 score=0.8891 reg_scale=1.00 beta_std=0.3815 beta_range=0.9975


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[Stage2 015] val auc_sev_cond=0.8450 val auc_sick=0.9297 score=0.8915 reg_scale=1.00 beta_std=0.3849 beta_range=0.9966
Early stopping at stage2 epoch 15
W drift (fro): 0.0
saved /content/drive/MyDrive/scMILD_final/runs/gorV3/checkpoints/last_ckpt.pt
saved /content/drive/MyDrive/scMILD_final/runs/gorV3/checkpoints/best_ckpt.pt
saved /content/drive/MyDrive/scMILD_final/runs/gorV3/history.json


In [19]:

# Step 11. Generic prediction exporter (supports best/last file separation)
@torch.no_grad()
def run_model_batch(model, batch, device):
    model.eval()
    batch_dev = to_device_batch(batch, device)
    out = model(batch_dev["x"], batch_dev["mask"])

    p3 = out["p_3"].detach().cpu().numpy()
    p_sick = out["p_sick"].detach().cpu().numpy()
    p_sev_cond = out["p_sev_cond"].detach().cpu().numpy()
    return out, p3, p_sick, p_sev_cond

@torch.no_grad()
def export_split_predictions(model, loader, device, split_name: str, save_dir: str, file_stem: Optional[str] = None):
    model.eval()
    rows = []
    example_out_keys = None

    for batch in tqdm(loader, desc=f"Predict {split_name}"):
        out, p3, p_sick, p_sev_cond = run_model_batch(model, batch, device)
        if example_out_keys is None:
            example_out_keys = list(out.keys())

        for i in range(len(batch["bag_id"])):
            y_true_i = int(batch["y"][i].item())
            y_pred_i = int(np.argmax(p3[i]))
            rows.append({
                "bag_id": str(batch["bag_id"][i]),
                "sample_id": str(batch["sample_id"][i]),
                "donor_id": str(batch["donor_id"][i]),
                "y_true_int": y_true_i,
                "y_true_str": str(batch["y_norm"][i]),
                "p_H": float(p3[i,0]),
                "p_M": float(p3[i,1]),
                "p_S": float(p3[i,2]),
                "p_sick": float(p_sick[i]),
                "p_sev_cond": float(p_sev_cond[i]),
                "y_pred_int": y_pred_i,
                "y_pred_str": INT_TO_LABEL[y_pred_i],
            })

    pred_df = pd.DataFrame(rows)

    # Integrity checks
    required_cols = ["bag_id","donor_id","y_true_int","p_H","p_M","p_S","p_sick","p_sev_cond","y_pred_int"]
    missing = [c for c in required_cols if c not in pred_df.columns]
    assert not missing, f"Missing columns: {missing}"

    prob_sum = pred_df[["p_H","p_M","p_S"]].sum(axis=1).to_numpy()
    max_dev = float(np.abs(prob_sum - 1.0).max())
    print(f"[{split_name}] max |p_H+p_M+p_S-1| = {max_dev:.3e}")
    assert max_dev < 1e-4 + 1e-6, "3-class probs do not sum to 1"

    assert not pred_df["bag_id"].duplicated().any(), f"Duplicated bag_id in {split_name} export"
    assert len(pred_df) == len(loader.dataset), f"{split_name}: pred rows != n_bags"

    os.makedirs(save_dir, exist_ok=True)
    fname = file_stem if file_stem is not None else f"{split_name}_predictions.csv"
    save_path = os.path.join(save_dir, fname)
    pred_df.to_csv(save_path, index=False)

    print(f"[{split_name}] saved: {save_path} | shape={pred_df.shape}")
    print(f"[{split_name}] example model output keys: {example_out_keys}")
    return pred_df, save_path

@torch.no_grad()
def export_bag_latents(model, loader, device, out_path: str):
    model.eval()
    rows = []
    for batch in tqdm(loader, desc="Dump bag latents"):
        batch_dev = to_device_batch(batch, device)
        out = model(batch_dev["x"], batch_dev["mask"])

        z_d = out["z_d"].detach().cpu().numpy()
        z_s = out["z_s"].detach().cpu().numpy()
        a = out["a"].detach().cpu().numpy()
        r = out["r"].detach().cpu().numpy()
        a_keep = out["a_keep"].detach().cpu().numpy()
        p3 = out["p_3"].detach().cpu().numpy()
        p_sick = out["p_sick"].detach().cpu().numpy()
        p_sev_cond = out["p_sev_cond"].detach().cpu().numpy()

        for i in range(len(batch["bag_id"])):
            rows.append({
                "bag_id": str(batch["bag_id"][i]),
                "sample_id": str(batch["sample_id"][i]),
                "donor_id": str(batch["donor_id"][i]),
                "y_true_int": int(batch["y"][i].item()),
                "y_true_str": str(batch["y_norm"][i]),
                "p_H": float(p3[i,0]),
                "p_M": float(p3[i,1]),
                "p_S": float(p3[i,2]),
                "p_sick": float(p_sick[i]),
                "p_sev_cond": float(p_sev_cond[i]),
                "z_d": z_d[i].tolist(),
                "z_s": z_s[i].tolist(),
                "a_coef": a[i].tolist(),
                "r": r[i].tolist(),
                "a_keep": a_keep[i].tolist(),
            })
    df = pd.DataFrame(rows)
    df.to_json(out_path, orient="records")
    print("Saved bag-level latents:", out_path, "| n=", len(df))
    return df

print("Exporter utilities ready.")

# Integrity test on current (trained-last) state for val
_ = export_split_predictions(model, val_loader, DEVICE, "val_sanity", save_dir=OUT_DIR, file_stem="_tmp_val_sanity.csv")
os.remove(os.path.join(OUT_DIR, "_tmp_val_sanity.csv"))
print("Step 11 integrity check passed.")


Exporter utilities ready.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predict val_sanity:   0%|          | 0/6 [00:00<?, ?it/s]

[val_sanity] max |p_H+p_M+p_S-1| = 4.470e-08
[val_sanity] saved: /content/drive/MyDrive/scMILD_final/runs/gorV3/eval_artifacts/_tmp_val_sanity.csv | shape=(42, 12)
[val_sanity] example model output keys: ['z_d', 'z_s', 'attn_d', 'attn_s', 'score_d', 'score_s', 'p_sick', 'logits_each', 'p_each', 'proto_idx', 'W', 'Q', 'R', 'z_prog', 'a', 'a_keep', 'z_parallel', 'r', 'beta', 'beta_global', 'beta_logit', 'sev_logit', 'p_sev_cond', 'p_H', 'p_M', 'p_S', 'p_3', 'query_orth_pen']
Step 11 integrity check passed.


In [22]:
# Recovery Cell (replaces Step 12~15): file-based exports + eval + summary (restart-safe)
# UPDATED: supports conditional beta (bag-conditional gate) checkpoints

import os, json
import numpy as np
import pandas as pd
import torch
from dataclasses import asdict
from tqdm.auto import tqdm

# -------------------------
# 0) Hard prereqs
# -------------------------
for _k in ["RUN_DIR", "OUT_DIR", "CKPT_DIR", "DEVICE"]:
    if _k not in globals():
        raise RuntimeError(f"Missing {_k}. Run Step 0~1 first.")

for _k in ["GORHierMIL", "ModelConfig", "TrainConfig"]:
    if _k not in globals():
        raise RuntimeError(f"Missing {_k}. Run Step 7~9 first (model/util definitions).")

for _k in ["val_loader", "test_loader", "train_loader"]:
    if _k not in globals():
        raise RuntimeError(f"Missing {_k}. Run Step 2~6 first (data + loaders).")

# sklearn optional (don’t crash if missing)
SKLEARN_OK = bool(globals().get("SKLEARN_OK", False))
if SKLEARN_OK:
    try:
        from sklearn.metrics import (
            roc_auc_score, average_precision_score, balanced_accuracy_score,
            accuracy_score, f1_score, confusion_matrix
        )
    except Exception:
        SKLEARN_OK = False

# exporter utilities; Step 11 defines them, but we provide fallbacks if missing
if "to_device_batch" not in globals():
    def to_device_batch(batch: dict, device: str) -> dict:
        out = {}
        for k, v in batch.items():
            out[k] = v.to(device, non_blocking=True) if torch.is_tensor(v) else v
        return out

# -------------------------
# 0.5) Conditional-GOR availability check (only required when ckpt is conditional)
# -------------------------
_HAS_COND_GOR = ("ConditionalGatedOrthogonalResidual" in globals())

# -------------------------
# 1) Load checkpoints + history from disk
# -------------------------
best_ckpt_path = os.path.join(CKPT_DIR, "best_ckpt.pt")
last_ckpt_path = os.path.join(CKPT_DIR, "last_ckpt.pt")
if not os.path.exists(best_ckpt_path):
    raise FileNotFoundError(f"Missing: {best_ckpt_path} (run Step 10 first)")
if not os.path.exists(last_ckpt_path):
    raise FileNotFoundError(f"Missing: {last_ckpt_path} (run Step 10 first)")

best_ckpt = torch.load(best_ckpt_path, map_location="cpu")
last_ckpt = torch.load(last_ckpt_path, map_location="cpu")

history_path = os.path.join(RUN_DIR, "history.json")
history = []
if os.path.exists(history_path):
    with open(history_path, "r") as f:
        history = json.load(f)

# -------------------------
# 2) Rebuild models from checkpoint configs (prevents cfg drift bugs)
# -------------------------
from dataclasses import fields

def _dc_from_dict(cls, d: dict):
    keys = {f.name for f in fields(cls)}
    return cls(**{k: d[k] for k in d.keys() if k in keys})

best_cfg = _dc_from_dict(ModelConfig, best_ckpt["model_cfg"])
best_train_cfg = _dc_from_dict(TrainConfig, best_ckpt["train_cfg"])

last_cfg = _dc_from_dict(ModelConfig, last_ckpt["model_cfg"])
last_train_cfg = _dc_from_dict(TrainConfig, last_ckpt["train_cfg"])

def _is_cond_ckpt(state_dict: dict) -> bool:
    # signature: conditional gate params exist
    return any(k.startswith("gor.gate.") for k in state_dict.keys())

def _patch_gor_by_ckpt(m, cfg: ModelConfig, state_dict: dict):
    is_cond = _is_cond_ckpt(state_dict)
    if is_cond:
        if not _HAS_COND_GOR:
            raise RuntimeError(
                "Checkpoint appears to be conditional (has gor.gate.*), but "
                "ConditionalGatedOrthogonalResidual is not defined in this runtime. "
                "Run the conditional-gor definition cell first."
            )
        # swap gor to conditional version BEFORE load_state_dict
        m.gor = ConditionalGatedOrthogonalResidual(
            d_branch=int(cfg.d_branch),
            K_proto=int(cfg.K_proto),
            hidden=cfg.sev_head_hidden,
            dropout=cfg.dropout,
            beta_init=cfg.beta_init,
        ).to(next(m.parameters()).device)
    return is_cond

def _load_model(cfg: ModelConfig, state_dict: dict):
    m = GORHierMIL(cfg).to(DEVICE)
    is_cond = _patch_gor_by_ckpt(m, cfg, state_dict)
    # strict load: if mismatch, fail loudly (good)
    m.load_state_dict(state_dict, strict=True)
    m.eval()
    return m, is_cond

best_model, best_is_cond = _load_model(best_cfg, best_ckpt["model_state_dict"])
last_model, last_is_cond = _load_model(last_cfg, last_ckpt["model_state_dict"])

# quick sanity: print beta shape using one batch
best_model.eval()
b = next(iter(train_loader))
b = to_device_batch(b, DEVICE)
with torch.no_grad():
    out = best_model(b["x"], b["mask"])
print("[Sanity] best ckpt conditional?:", bool(best_is_cond))
print("[Sanity] beta shape:", tuple(out["beta"].shape))  # cond: (B,K), non-cond: (K,) or (B,K) depending on old impl

# -------------------------
# 3) Export predictions (fallbacks if missing)
# -------------------------
if "export_split_predictions" not in globals():
    @torch.no_grad()
    def run_model_batch(model, batch, device):
        model.eval()
        batch_dev = to_device_batch(batch, device)
        out = model(batch_dev["x"], batch_dev["mask"])
        p3 = out["p_3"].detach().cpu().numpy()
        p_sick = out["p_sick"].detach().cpu().numpy()
        p_sev_cond = out["p_sev_cond"].detach().cpu().numpy()
        return out, p3, p_sick, p_sev_cond

    @torch.no_grad()
    def export_split_predictions(model, loader, device, split_name: str, save_dir: str, file_stem: str):
        model.eval()
        rows = []
        for batch in tqdm(loader, desc=f"Predict {split_name}"):
            out, p3, p_sick, p_sev_cond = run_model_batch(model, batch, device)
            for i in range(len(batch["bag_id"])):
                y_true_i = int(batch["y"][i].item()) if "y" in batch else int(batch["y_int"][i])
                y_pred_i = int(np.argmax(p3[i]))
                if "INT_TO_LABEL" in globals():
                    y_pred_str = INT_TO_LABEL[y_pred_i]
                else:
                    y_pred_str = {0:"control", 1:"mild/moderate", 2:"severe/critical"}.get(y_pred_i, str(y_pred_i))

                rows.append({
                    "bag_id": str(batch["bag_id"][i]),
                    "sample_id": str(batch.get("sample_id", batch["bag_id"])[i]),
                    "donor_id": str(batch["donor_id"][i]),
                    "y_true_int": y_true_i,
                    "y_true_str": str(batch["y_norm"][i]) if "y_norm" in batch else str(y_true_i),
                    "p_H": float(p3[i,0]),
                    "p_M": float(p3[i,1]),
                    "p_S": float(p3[i,2]),
                    "p_sick": float(p_sick[i]),
                    "p_sev_cond": float(p_sev_cond[i]),
                    "y_pred_int": y_pred_i,
                    "y_pred_str": y_pred_str,
                })
        pred_df = pd.DataFrame(rows)
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, file_stem)
        pred_df.to_csv(save_path, index=False)
        return pred_df, save_path

# -------------------------
# 3.5) Export bag latents (UPDATED: includes beta per bag)
# -------------------------
if "export_bag_latents" not in globals():
    @torch.no_grad()
    def export_bag_latents(model, loader, device, out_path: str):
        model.eval()
        rows = []
        for batch in tqdm(loader, desc="Dump bag latents"):
            batch_dev = to_device_batch(batch, device)
            out = model(batch_dev["x"], batch_dev["mask"])

            z_d = out["z_d"].detach().cpu().numpy()
            z_s = out["z_s"].detach().cpu().numpy()
            a = out["a"].detach().cpu().numpy()
            r = out["r"].detach().cpu().numpy()
            a_keep = out["a_keep"].detach().cpu().numpy()
            p3 = out["p_3"].detach().cpu().numpy()
            p_sick = out["p_sick"].detach().cpu().numpy()
            p_sev_cond = out["p_sev_cond"].detach().cpu().numpy()

            beta = out["beta"].detach().cpu().numpy()
            # normalize beta shape to per-bag vectors
            # - conditional: beta is [B,K]
            # - legacy: could be [K] or [B,K]
            if beta.ndim == 1:
                beta = np.repeat(beta[None, :], repeats=len(batch["bag_id"]), axis=0)

            for i in range(len(batch["bag_id"])):
                rows.append({
                    "bag_id": str(batch["bag_id"][i]),
                    "sample_id": str(batch.get("sample_id", batch["bag_id"])[i]),
                    "donor_id": str(batch["donor_id"][i]),
                    "y_true_int": int(batch["y"][i].item()) if "y" in batch else int(batch["y_int"][i]),
                    "y_true_str": str(batch["y_norm"][i]) if "y_norm" in batch else "",
                    "p_H": float(p3[i,0]),
                    "p_M": float(p3[i,1]),
                    "p_S": float(p3[i,2]),
                    "p_sick": float(p_sick[i]),
                    "p_sev_cond": float(p_sev_cond[i]),
                    "z_d": z_d[i].tolist(),
                    "z_s": z_s[i].tolist(),
                    "a_coef": a[i].tolist(),
                    "r": r[i].tolist(),
                    "a_keep": a_keep[i].tolist(),
                    "beta": beta[i].tolist(),                 # NEW
                    "beta_mean": float(beta[i].mean()),       # NEW (handy)
                    "beta_std": float(beta[i].std()),         # NEW
                })
        df = pd.DataFrame(rows)
        df.to_json(out_path, orient="records")
        return df

# -------------------------
# 3.6) Helper: collect beta arrays over a loader (UPDATED)
# -------------------------
@torch.no_grad()
def collect_beta_over_loader(model, loader, device):
    model.eval()
    betas = []
    for batch in tqdm(loader, desc="Collect beta"):
        batch_dev = to_device_batch(batch, device)
        out = model(batch_dev["x"], batch_dev["mask"])
        beta = out["beta"].detach().cpu().numpy()
        if beta.ndim == 1:
            # legacy global beta -> expand to per-bag rows
            beta = np.repeat(beta[None, :], repeats=len(batch["bag_id"]), axis=0)
        betas.append(beta)
    beta_all = np.concatenate(betas, axis=0)  # [N_bags, K]
    return beta_all

# -------------------------
# 4) Export predictions
# -------------------------
val_pred_best_df, val_pred_best_path = export_split_predictions(
    best_model, val_loader, DEVICE, "val", OUT_DIR, file_stem="val_predictions_best.csv"
)
test_pred_best_df, test_pred_best_path = export_split_predictions(
    best_model, test_loader, DEVICE, "test", OUT_DIR, file_stem="test_predictions_best.csv"
)
val_pred_last_df, val_pred_last_path = export_split_predictions(
    last_model, val_loader, DEVICE, "val", OUT_DIR, file_stem="val_predictions_last.csv"
)
test_pred_last_df, test_pred_last_path = export_split_predictions(
    last_model, test_loader, DEVICE, "test", OUT_DIR, file_stem="test_predictions_last.csv"
)

# -------------------------
# 5) Save beta arrays (BEST is canonical)
#     UPDATED: supports conditional beta => save per-bag [N,K] + global mean [K]
# -------------------------
beta_best_all = collect_beta_over_loader(best_model, test_loader, DEVICE)  # [N_bags, K]
beta_last_all = collect_beta_over_loader(last_model, test_loader, DEVICE)

# basic integrity: K match
K_best = int(best_cfg.K_proto)
K_last = int(last_cfg.K_proto)
if beta_best_all.shape[1] != K_best:
    raise RuntimeError(f"beta_best_all shape mismatch: {beta_best_all.shape} vs K_proto={K_best}")
if beta_last_all.shape[1] != K_last:
    raise RuntimeError(f"beta_last_all shape mismatch: {beta_last_all.shape} vs K_proto={K_last}")

beta_best_global = beta_best_all.mean(axis=0)  # [K]
beta_last_global = beta_last_all.mean(axis=0)

# save per-bag
np.save(os.path.join(RUN_DIR, "beta_values_best.npy"), beta_best_all)
np.save(os.path.join(RUN_DIR, "beta_values_last.npy"), beta_last_all)
np.save(os.path.join(RUN_DIR, "beta_values.npy"), beta_best_all)  # canonical (per-bag)

# save global (handy for quick plots)
np.save(os.path.join(RUN_DIR, "beta_values_best_global.npy"), beta_best_global)
np.save(os.path.join(RUN_DIR, "beta_values_last_global.npy"), beta_last_global)
np.save(os.path.join(RUN_DIR, "beta_values_global.npy"), beta_best_global)  # canonical global

_ = export_bag_latents(best_model, test_loader, DEVICE, os.path.join(RUN_DIR, "test_bag_latents_best.json"))

# -------------------------
# 6) Eval metrics: bag-level + donor-level mean aggregation
# -------------------------
INT_TO_LABEL_LOCAL = globals().get("INT_TO_LABEL", {0:"control", 1:"mild/moderate", 2:"severe/critical"})

def compute_metrics_from_pred_df(df: pd.DataFrame) -> dict:
    y = df["y_true_int"].to_numpy().astype(int)
    yhat = df["y_pred_int"].to_numpy().astype(int)

    out = {"n": int(len(df))}
    out["acc_3class"] = float((y == yhat).mean()) if not SKLEARN_OK else float(accuracy_score(y, yhat))

    if SKLEARN_OK:
        out["bacc_3class"] = float(balanced_accuracy_score(y, yhat))
        out["macro_f1_3class"] = float(f1_score(y, yhat, average="macro"))
        out["confusion_3class"] = confusion_matrix(y, yhat, labels=[0,1,2]).tolist()

        y_sick = (y > 0).astype(int)
        p_sick = df["p_sick"].to_numpy()
        if len(np.unique(y_sick)) > 1:
            out["auc_sick"] = float(roc_auc_score(y_sick, p_sick))
            out["aupr_sick"] = float(average_precision_score(y_sick, p_sick))

        m = (y > 0)
        if m.sum() > 0:
            y_sev = (y[m] == 2).astype(int)
            p_sev_cond = df.loc[m, "p_sev_cond"].to_numpy()
            if len(np.unique(y_sev)) > 1:
                out["auc_sev_cond"] = float(roc_auc_score(y_sev, p_sev_cond))
                out["aupr_sev_cond"] = float(average_precision_score(y_sev, p_sev_cond))
    return out

def aggregate_predictions_to_donor(df: pd.DataFrame, agg: str = "mean") -> pd.DataFrame:
    reducer = np.mean if agg == "mean" else np.max
    rows = []
    for donor_id, g in df.groupby("donor_id", sort=False):
        uniq = g["y_true_int"].unique()
        if len(uniq) != 1:
            raise RuntimeError(f"Donor {donor_id} has mixed labels: {uniq}")

        row = {
            "donor_id": str(donor_id),
            "n_bags": int(len(g)),
            "y_true_int": int(uniq[0]),
            "y_true_str": str(g["y_true_str"].iloc[0]),
            "p_H": float(reducer(g["p_H"].to_numpy())),
            "p_M": float(reducer(g["p_M"].to_numpy())),
            "p_S": float(reducer(g["p_S"].to_numpy())),
            "p_sick": float(reducer(g["p_sick"].to_numpy())),
            "p_sev_cond": float(reducer(g["p_sev_cond"].to_numpy())),
        }
        rows.append(row)

    ddf = pd.DataFrame(rows)
    ddf["y_pred_int"] = ddf[["p_H","p_M","p_S"]].to_numpy().argmax(axis=1)
    ddf["y_pred_str"] = ddf["y_pred_int"].map(INT_TO_LABEL_LOCAL)
    return ddf

eval_files = {
    "val_best": val_pred_best_path,
    "test_best": test_pred_best_path,
    "val_last": val_pred_last_path,
    "test_last": test_pred_last_path,
}

all_metrics = {}
for name, path in eval_files.items():
    df = pd.read_csv(path)
    bag_metrics = compute_metrics_from_pred_df(df)
    donor_df = aggregate_predictions_to_donor(df, agg="mean")
    donor_metrics = compute_metrics_from_pred_df(donor_df)

    donor_pred_path = os.path.join(OUT_DIR, f"{name}_donor_predictions.csv")
    donor_df.to_csv(donor_pred_path, index=False)

    all_metrics[name] = {
        "bag": bag_metrics,
        "donor_meanagg": donor_metrics,
        "paths": {"bag_pred_csv": path, "donor_pred_csv": donor_pred_path},
    }

with open(os.path.join(OUT_DIR, "metrics_bag_and_donor.json"), "w") as f:
    json.dump(all_metrics, f, indent=2)

# -------------------------
# 7) Consistency checks vs metadata (from disk, not memory)
# -------------------------
def _normalize_label(s: str) -> str:
    s = str(s).strip().lower()
    s = s.replace(" ", "").replace("_", "").replace("-", "")
    return s

bags_all_path = os.path.join(RUN_DIR, "bags_all.csv")
bags_all_df = None
if "bags_all" in globals() and isinstance(globals()["bags_all"], pd.DataFrame):
    bags_all_df = globals()["bags_all"].copy()
elif os.path.exists(bags_all_path):
    bags_all_df = pd.read_csv(bags_all_path)

def validate_pred_vs_meta(pred_df: pd.DataFrame, bag_meta_df: pd.DataFrame, tag: str):
    merged = pred_df.merge(
        bag_meta_df[["bag_id","sample_id","donor_id","y_true_str","y_true_int"]],
        on="bag_id", how="left", suffixes=("_pred", "_meta")
    )
    n_missing = int(merged["sample_id_meta"].isna().sum()) if "sample_id_meta" in merged.columns else 0
    if n_missing != 0:
        raise RuntimeError(f"{tag}: metadata join failed; missing rows={n_missing}")

    if "y_true_int_pred" in merged.columns and "y_true_int_meta" in merged.columns:
        mismatch_int = int((merged["y_true_int_pred"] != merged["y_true_int_meta"]).sum())
        if mismatch_int != 0:
            raise RuntimeError(f"{tag}: y_true_int mismatch rows={mismatch_int}")

    if "y_true_str_pred" in merged.columns and "y_true_str_meta" in merged.columns:
        mismatch_str = int((
            merged["y_true_str_pred"].astype(str).map(_normalize_label) !=
            merged["y_true_str_meta"].astype(str).map(_normalize_label)
        ).sum())
        if mismatch_str != 0:
            raise RuntimeError(f"{tag}: y_true_str mismatch rows={mismatch_str}")

    if "donor_id_pred" in merged.columns and "donor_id_meta" in merged.columns:
        donor_mismatch = int((merged["donor_id_pred"].astype(str) != merged["donor_id_meta"].astype(str)).sum())
        if donor_mismatch != 0:
            raise RuntimeError(f"{tag}: donor_id mismatch rows={donor_mismatch}")

    return merged

if bags_all_df is not None:
    bag_val_meta  = bags_all_df[bags_all_df["split"] == "val"].copy()
    bag_test_meta = bags_all_df[bags_all_df["split"] == "test"].copy()
    _ = validate_pred_vs_meta(val_pred_best_df, bag_val_meta, "val_best")
    _ = validate_pred_vs_meta(test_pred_best_df, bag_test_meta, "test_best")

# -------------------------
# 8) Final summary + manifest (no in-memory best/history assumptions)
# -------------------------
if "eval_one_epoch" not in globals():
    raise RuntimeError("Missing eval_one_epoch. Run Step 9 first.")

best_val_logs_re, best_val_metrics_re = eval_one_epoch(best_model, val_loader, epoch=999, cfg=best_train_cfg, stage=2, device=DEVICE)
best_test_logs_re, best_test_metrics_re = eval_one_epoch(best_model, test_loader, epoch=999, cfg=best_train_cfg, stage=2, device=DEVICE)

best_epoch = int(best_ckpt.get("epoch2", -1))
best_score = float(best_ckpt.get("best_score", np.nan))
last_epoch = int(last_ckpt.get("epoch2", -1))
selection_formula = best_ckpt.get("selection_formula", globals().get("SELECTION_FORMULA_STR", ""))

best_history_row = None
for r in history:
    if int(r.get("stage", -1)) == 2 and int(r.get("epoch2", -1)) == best_epoch:
        best_history_row = r
        break

data_cfg_obj = globals().get("DATA_CFG", None)
data_cfg = asdict(data_cfg_obj) if data_cfg_obj is not None else None

summary = {
    "exp_id": best_ckpt.get("exp_id", globals().get("EXP_ID", "")),
    "run_dir": RUN_DIR,
    "out_dir": OUT_DIR,
    "ckpt_dir": CKPT_DIR,
    "selection_formula": selection_formula,
    "best_epoch": best_epoch,
    "best_score": best_score,
    "last_epoch": last_epoch,
    "best_history_row": best_history_row,

    "best_val_metrics": best_ckpt.get("best_val_metrics", None),
    "best_val_logs": best_ckpt.get("best_val_logs", None),

    "best_val_metrics_reloaded": best_val_metrics_re,
    "best_val_logs_reloaded": best_val_logs_re,
    "test_metrics_best_reloaded": best_test_metrics_re,
    "test_logs_best_reloaded": best_test_logs_re,

    # UPDATED beta info
    "beta_is_conditional_ckpt": bool(best_is_cond),
    "beta_values_global": beta_best_global.tolist(),         # [K]
    "beta_values_perbag_shape": list(beta_best_all.shape),   # [N, K]
    "beta_stats_global": {
        "mean": float(beta_best_global.mean()),
        "std": float(beta_best_global.std()),
        "min": float(beta_best_global.min()),
        "max": float(beta_best_global.max()),
        "range": float(beta_best_global.max() - beta_best_global.min()),
    },
    "beta_stats_over_bags": {
        "mean": float(beta_best_all.mean()),
        "std": float(beta_best_all.std()),
        "min": float(beta_best_all.min()),
        "max": float(beta_best_all.max()),
        "range": float(beta_best_all.max() - beta_best_all.min()),
    },

    "model_cfg": asdict(best_cfg),
    "train_cfg": asdict(best_train_cfg),
    "data_cfg": data_cfg,

    "artifact_paths": {
        "history_json": history_path,
        "best_ckpt": best_ckpt_path,
        "last_ckpt": last_ckpt_path,
        "val_predictions_best": val_pred_best_path,
        "test_predictions_best": test_pred_best_path,
        "val_predictions_last": val_pred_last_path,
        "test_predictions_last": test_pred_last_path,
        "metrics_bag_and_donor_json": os.path.join(OUT_DIR, "metrics_bag_and_donor.json"),

        # UPDATED beta artifacts
        "beta_values_best_npy": os.path.join(RUN_DIR, "beta_values_best.npy"),           # [N,K]
        "beta_values_last_npy": os.path.join(RUN_DIR, "beta_values_last.npy"),           # [N,K]
        "beta_values_npy_canonical": os.path.join(RUN_DIR, "beta_values.npy"),           # [N,K]
        "beta_values_best_global_npy": os.path.join(RUN_DIR, "beta_values_best_global.npy"),  # [K]
        "beta_values_last_global_npy": os.path.join(RUN_DIR, "beta_values_last_global.npy"),  # [K]
        "beta_values_global_npy_canonical": os.path.join(RUN_DIR, "beta_values_global.npy"),  # [K]

        "test_bag_latents_best_json": os.path.join(RUN_DIR, "test_bag_latents_best.json"),
        "bags_all_csv": bags_all_path if os.path.exists(bags_all_path) else None,
    },
}

summary_path = os.path.join(RUN_DIR, "summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

manifest = pd.DataFrame([{
    "exp_id": summary["exp_id"],
    "best_epoch": summary["best_epoch"],
    "best_score": summary["best_score"],
    "beta_global_mean": summary["beta_stats_global"]["mean"],
    "beta_global_std": summary["beta_stats_global"]["std"],
    "beta_global_range": summary["beta_stats_global"]["range"],
    "beta_all_mean": summary["beta_stats_over_bags"]["mean"],
    "beta_all_std": summary["beta_stats_over_bags"]["std"],
    "beta_all_range": summary["beta_stats_over_bags"]["range"],
    "val_bacc_best": (summary["best_val_metrics"] or {}).get("bacc_3class", np.nan),
    "val_auc_sick_best": (summary["best_val_metrics"] or {}).get("auc_sick", np.nan),
    "val_auc_sev_cond_best": (summary["best_val_metrics"] or {}).get("auc_sev_cond", np.nan),
    "test_auc_sick_best_reload": summary["test_metrics_best_reloaded"].get("auc_sick", np.nan),
    "test_auc_sev_cond_best_reload": summary["test_metrics_best_reloaded"].get("auc_sev_cond", np.nan),
}])
manifest_path = os.path.join(RUN_DIR, "run_manifest.csv")
manifest.to_csv(manifest_path, index=False)

# -------------------------
# 9) Integrity
# -------------------------
expected = [
    summary_path, manifest_path,
    os.path.join(OUT_DIR, "metrics_bag_and_donor.json"),
    val_pred_best_path, test_pred_best_path, val_pred_last_path, test_pred_last_path,
    os.path.join(RUN_DIR, "beta_values_best.npy"),
    os.path.join(RUN_DIR, "beta_values_last.npy"),
    os.path.join(RUN_DIR, "beta_values.npy"),
    os.path.join(RUN_DIR, "beta_values_best_global.npy"),
    os.path.join(RUN_DIR, "beta_values_last_global.npy"),
    os.path.join(RUN_DIR, "beta_values_global.npy"),
    os.path.join(RUN_DIR, "test_bag_latents_best.json"),
]
for p in expected:
    if not os.path.exists(p):
        raise RuntimeError(f"Missing expected artifact: {p}")

print("[OK] Recovery cell finished.")
print("summary.json :", summary_path)
print("manifest.csv :", manifest_path)
print("metrics json :", os.path.join(OUT_DIR, "metrics_bag_and_donor.json"))
print("beta (per-bag) canonical :", os.path.join(RUN_DIR, "beta_values.npy"))
print("beta (global)  canonical :", os.path.join(RUN_DIR, "beta_values_global.npy"))

[Sanity] best ckpt conditional?: True
[Sanity] beta shape: (8, 12)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predict val:   0%|          | 0/6 [00:00<?, ?it/s]

[val] max |p_H+p_M+p_S-1| = 4.470e-08
[val] saved: /content/drive/MyDrive/scMILD_final/runs/gorV3/eval_artifacts/val_predictions_best.csv | shape=(42, 12)
[val] example model output keys: ['z_d', 'z_s', 'attn_d', 'attn_s', 'score_d', 'score_s', 'p_sick', 'logits_each', 'p_each', 'proto_idx', 'W', 'Q', 'R', 'z_prog', 'a', 'a_keep', 'z_parallel', 'r', 'beta', 'beta_global', 'beta_logit', 'sev_logit', 'p_sev_cond', 'p_H', 'p_M', 'p_S', 'p_3', 'query_orth_pen']


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Collect beta:   0%|          | 0/6 [00:00<?, ?it/s]

Dump bag latents:   0%|          | 0/6 [00:00<?, ?it/s]

Saved bag-level latents: /content/drive/MyDrive/scMILD_final/runs/gorV3/test_bag_latents_best.json | n= 42


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[OK] Recovery cell finished.
summary.json : /content/drive/MyDrive/scMILD_final/runs/gorV3/summary.json
manifest.csv : /content/drive/MyDrive/scMILD_final/runs/gorV3/run_manifest.csv
metrics json : /content/drive/MyDrive/scMILD_final/runs/gorV3/eval_artifacts/metrics_bag_and_donor.json
beta (per-bag) canonical : /content/drive/MyDrive/scMILD_final/runs/gorV3/beta_values.npy
beta (global)  canonical : /content/drive/MyDrive/scMILD_final/runs/gorV3/beta_values_global.npy


## Step 16 (optional next experiments)
이 노트북은 **단일 실험 실행 + best/last export + donor-level 평가**까지 포함한다.

다음 실험은 이 노트북을 복사해서 다음 변수만 바꿔서 반복:
- `EXP_ID`
- `MODEL_CFG.K_proto`
- `TRAIN_CFG.lambda_beta_center`
- `TRAIN_CFG.lambda_beta_spread`, `TRAIN_CFG.lambda_beta_range`
- `TRAIN_CFG.beta_target_std`, `TRAIN_CFG.beta_target_range`

권장 그리드(최소):
- K = 4, 8, 12, 16
- lambda_beta_center = 0 or 1e-5
- lambda_beta_spread = 1e-3, 1e-2
- lambda_beta_range = 1e-3, 1e-2
